### **2.3.1. Load and audit evidence candidates.**


In [122]:
import json, re
import ast
import pandas as pd
from pathlib import Path
from typing import Literal
from pydantic import BaseModel, ConfigDict

import warnings
warnings.filterwarnings("ignore")  # suppress all warnings

In [44]:
# Load the finalized Step 2.2 outputs without reparsing documents
PROJECT_ROOT = Path("/Users/tanggiee/Desktop/RAG_AI/esg_rag_project")
PARSED_FOLDER = PROJECT_ROOT / "data" / "parsed"
SPLITS = ["development", "eval", "test"]

def load_jsonl(path):
    with path.open(encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]

units_df = pd.DataFrame([record for split in SPLITS for record in load_jsonl(PARSED_FOLDER / f"{split}_units.jsonl")])
provisions_df = pd.DataFrame([record for split in SPLITS for record in load_jsonl(PARSED_FOLDER / f"{split}_provisions.jsonl")])

print("Units:", len(units_df))
print("Provisions:", len(provisions_df))
display(units_df.groupby(["split", "unit_type"]).size().unstack(fill_value=0))
display(provisions_df.groupby(["split", "provision_type"]).size().unstack(fill_value=0))

Units: 11013
Provisions: 87693


unit_type,article,document_fallback,roman_section
split,,,
development,7827,3,17
eval,1380,0,15
test,1771,0,0


provision_type,clause,embedded_article,lettered_item,numbered_item,point
split,,,,,
development,31271,361,12,66,32866
eval,5207,92,0,66,4547
test,6845,116,0,0,6244


In [45]:
# Standardise parent units as evidence records
unit_candidates = units_df.copy()
unit_candidates["evidence_id"] = unit_candidates["unit_id"]
unit_candidates["evidence_level"] = "unit"
unit_candidates["evidence_type"] = unit_candidates["unit_type"]
unit_candidates["source_article"] = unit_candidates["unit_number"]
unit_candidates["parent_article_number"] = unit_candidates["unit_number"]
unit_candidates["evidence_number"] = unit_candidates["unit_number"]
unit_candidates["evidence_title"] = unit_candidates["unit_title"]
unit_candidates["source_text"] = unit_candidates["unit_text"]
unit_candidates["evidence_start_char"] = unit_candidates["start_char"]
unit_candidates["evidence_end_char"] = unit_candidates["end_char"]

In [46]:
# Attach parent-Article metadata to child provisions
parent_metadata = units_df[["unit_id", "unit_number", "unit_title", "unit_type", "is_oversized"]].rename(columns={"unit_id": "parent_unit_id", "unit_number": "parent_article_number", "unit_title": "parent_article_title", "unit_type": "parent_unit_type", "is_oversized": "parent_is_oversized"})
provision_candidates = provisions_df.merge(parent_metadata, on="parent_unit_id", how="left", validate="many_to_one")

In [47]:
# Standardise child provisions as evidence records
provision_candidates["evidence_id"] = provision_candidates["provision_id"]
provision_candidates["evidence_level"] = "provision"
provision_candidates["evidence_type"] = provision_candidates["provision_type"]
provision_candidates["source_article"] = provision_candidates["parent_article_number"]
provision_candidates["evidence_number"] = provision_candidates["provision_number"]
provision_candidates["evidence_title"] = provision_candidates["provision_title"]
provision_candidates["source_text"] = provision_candidates["provision_text"]
provision_candidates["evidence_start_char"] = provision_candidates["start_char"]
provision_candidates["evidence_end_char"] = provision_candidates["end_char"]

In [48]:
# Combine both evidence levels without modifying the parser outputs
evidence_candidates_df = pd.concat([unit_candidates, provision_candidates], ignore_index=True, sort=False)

# Confirm that every record has a unique ID and every provision has a valid parent
assert evidence_candidates_df["evidence_id"].is_unique
assert provision_candidates["parent_article_number"].notna().all()

print("Combined evidence candidates:", len(evidence_candidates_df))

Combined evidence candidates: 98706


In [49]:
# Identify inactive legal-status records using headings or short status-only text
inactive_pattern = r"\b(annul(?:led|ment)?|repeal(?:ed|ment)?|abrogat(?:ed|ion)?)\b"
source_text = evidence_candidates_df["source_text"].fillna("")
# Use the standardised evidence title; short status-only text is checked separately
heading_text = evidence_candidates_df["evidence_title"].fillna("")

evidence_candidates_df["text_length"] = source_text.str.len()
evidence_candidates_df["is_inactive"] = heading_text.str.contains(inactive_pattern, case=False, regex=True) | (evidence_candidates_df["text_length"].le(200) & source_text.str.contains(inactive_pattern, case=False, regex=True))
evidence_candidates_df["is_fallback"] = evidence_candidates_df["evidence_type"].eq("document_fallback")
evidence_candidates_df["is_too_short"] = evidence_candidates_df["text_length"].lt(30) & ~evidence_candidates_df["is_inactive"]
evidence_candidates_df["requires_review"] = evidence_candidates_df["is_fallback"] | evidence_candidates_df["is_too_short"]
evidence_candidates_df["eligible"] = ~evidence_candidates_df["requires_review"]

# Prefer self-contained evidence over isolated subordinate Points
evidence_candidates_df["selection_priority"] = 4
evidence_candidates_df.loc[evidence_candidates_df["evidence_type"].isin(["clause", "embedded_article"]), "selection_priority"] = 1
evidence_candidates_df.loc[evidence_candidates_df["evidence_type"].isin(["article", "roman_section"]), "selection_priority"] = 2
evidence_candidates_df.loc[evidence_candidates_df["evidence_type"].isin(["point", "numbered_item", "lettered_item"]), "selection_priority"] = 3
evidence_candidates_df.loc[evidence_candidates_df["evidence_level"].eq("unit") & evidence_candidates_df["is_oversized"].fillna(False), "selection_priority"] = 4
evidence_candidates_df.loc[evidence_candidates_df["is_fallback"], "selection_priority"] = 9

print("Eligible candidates:", evidence_candidates_df["eligible"].sum())
print("Candidates requiring review:", evidence_candidates_df["requires_review"].sum())

Eligible candidates: 96967
Candidates requiring review: 1739


In [50]:
# Review coverage counts for every split without displaying eval or test text
eligible_df = evidence_candidates_df[evidence_candidates_df["eligible"]].copy()

display(eligible_df.groupby(["split", "evidence_level"]).size().unstack(fill_value=0))
display(eligible_df.groupby(["split", "evidence_type"]).size().unstack(fill_value=0))
display(eligible_df.groupby("split")["text_length"].agg(["count", "median", "mean", "max"]).round(1))
display(evidence_candidates_df.groupby(["split", "requires_review"]).size().unstack(fill_value=0))

evidence_level,provision,unit
split,,
development,63288,7844
eval,9763,1395
test,12906,1771


evidence_type,article,clause,embedded_article,lettered_item,numbered_item,point,roman_section
split,,,,,,,
development,7827,31064,361,12,66,31785,17
eval,1380,5180,92,0,66,4425,15
test,1771,6767,116,0,0,6023,0


,count,median,mean,max
split,,,,
development,71132,233.0,528.2,180336
eval,11158,224.0,487.3,63150
test,14677,233.0,559.8,330046


requires_review,False,True
split,,
development,71132,1291
eval,11158,149
test,14677,299


In [51]:
# Inspect reproducible development examples; keep eval and test text hidden during development
development_sample = eligible_df[eligible_df["split"].eq("development")].sample(min(15, eligible_df["split"].eq("development").sum()), random_state=42)
display(development_sample[["doc_id", "evidence_level", "evidence_type", "source_article", "text_length", "is_inactive", "selection_priority"]])

with pd.option_context("display.max_colwidth", 500):
    display(development_sample[["doc_id", "source_article", "evidence_type", "source_text"]])

,doc_id,evidence_level,evidence_type,source_article,text_length,is_inactive,selection_priority
53637,36_2020_ND-CP_m_440993,provision,point,22,152,False,3
27198,82_2015_QH13_m_283670,provision,point,41,89,False,3
17237,03_2026_TT-BNNMT_m_696487,provision,point,22,154,False,3
68499,308_2025_ND-CP_m_688502,provision,point,42,405,False,3
46953,47_2024_QH15_m_647693 (1),provision,point,56,118,False,3
17323,02_VBHN-BXD_m_613835,provision,point,7,101,False,3
56795,50_2020_ND-CP_m_441405,provision,point,12,457,False,3
74692,03_2022_ND-CP_m_504828,provision,clause,31,159,False,1
4475,15_VBHN-VPQH_m_636777,unit,article,24,204,False,2
3354,116_2025_QH15_m_688491,unit,article,5,5018,False,4


,doc_id,source_article,evidence_type,source_text
53637,36_2020_ND-CP_m_440993,22,point,a) carry out filling and sealing of the borehole/well if the violation prescribed in Point c Clause 1 and Point b Clause 3 of this Article is committed;
27198,82_2015_QH13_m_283670,41,point,a) Serve the purpose of National defense and security and perform state management tasks;
17237,03_2026_TT-BNNMT_m_696487,22,point,"a) Take charge of organizing the implementation in accordance with point a, clause 1; and point a, clause 2, Article 59 of the Law on Veterinary Medicine;"
68499,308_2025_ND-CP_m_688502,42,point,"c) Record confirming permission to send the national treasure that is an archived document having special value or a private archive having special value for national display, research or preservation for a certain period of time according to the Form No. 35 in the Appendix I enclosed with this Decree with regard to the national treasure specified in clause 6 Article 50 of the Law on Cultural Heritage;"
46953,47_2024_QH15_m_647693 (1),56,point,b) Records of presentation for approval for planning objectives and records of presentation for approval for planning;
17323,02_VBHN-BXD_m_613835,7,point,"k) Responsibility for establishment, management and use of the database of the local drainage system;"
56795,50_2020_ND-CP_m_441405,12,point,"d) Within 05 working days from the date of receipt of the appraisal result from the Ministry of Agriculture and Rural Development, the supervisory authority shall provide an explanation and submit the completed application to the Ministry of Agriculture and Rural Development. Within 05 working days from the date of receipt of the completed application, the Ministry of Agriculture and Rural Development shall consider and propose it to the Prime Minister;"
74692,03_2022_ND-CP_m_504828,31,clause,"3. A fine ranging from VND 5,000,000 to VND 10,000,000 shall be imposed for operating a motor vehicle on a dike against the regulations set out in the license."
4475,15_VBHN-VPQH_m_636777,24,article,"Article 24. Quantity of cigarettes in packs\nThree years after this Law takes effect, the number of cigarettes in a pack must not be fewer than 20, except for cigars and cigarettes manufactured for export."
3354,116_2025_QH15_m_688491,5,article,"Article 5. Prevention and combat against cyber espionage; protection of information classified as state secrets, work secrets, business secrets, personal secrets, family secrets, and private life in cyberspace\n1. Acts of cyber espionage; infringement on state secrets, work secrets, business secrets, personal secrets, family secrets, and private life in cyberspace include:\na) Appropriating, trading, seizing, or deliberately disclosing information classified as state secrets, work secrets, o..."


### **2.3.2. Evidence sampling**

In [52]:
# Prepare a diverse development pool without using eval or test evidence
development_pool = eligible_df[eligible_df["split"].eq("development")].copy()

# Convert multi-label ESG metadata into a simple label for sampling coverage
def primary_label(value):
    if isinstance(value, list):
        return value[0] if value else "Unclassified"
    return str(value).split("/")[0] if pd.notna(value) and str(value).strip() else "Unclassified"

development_pool["primary_esg_domain"] = development_pool["esg_domains"].apply(primary_label)
development_pool["document_type"] = development_pool["document_type"].fillna("Unknown")
development_pool["length_band"] = pd.cut(development_pool["text_length"], bins=[0, 199, 999, float("inf")], labels=["short", "medium", "long"])

# Assign mutually exclusive evidence groups
development_pool["sampling_group"] = "other"
development_pool.loc[development_pool["evidence_type"].eq("article"), "sampling_group"] = "article"
development_pool.loc[development_pool["evidence_type"].eq("point"), "sampling_group"] = "point"
development_pool.loc[development_pool["evidence_type"].eq("clause"), "sampling_group"] = "clause"
development_pool.loc[development_pool["evidence_type"].eq("embedded_article"), "sampling_group"] = "embedded_article"
development_pool.loc[development_pool["evidence_level"].eq("unit") & development_pool["is_oversized"].fillna(False), "sampling_group"] = "oversized_unit"
development_pool.loc[development_pool["is_inactive"], "sampling_group"] = "inactive"

# Retain at most one candidate of each type from the same source Article
development_pool = development_pool.sort_values(["selection_priority", "text_length"]).drop_duplicates(["sampling_group", "doc_id", "source_article"])
display(development_pool["sampling_group"].value_counts().rename("available_candidates").to_frame())

,available_candidates
sampling_group,
article,7322
clause,6993
point,3807
oversized_unit,443
inactive,157
embedded_article,103
other,18


In [53]:
# Development pilot quotas: precise evidence is emphasized without excluding difficult cases
PILOT_QUOTAS = {"clause": 20, "article": 12, "point": 8, "embedded_article": 8, "oversized_unit": 4, "inactive": 4, "other": 4}
print("Target pilot size:", sum(PILOT_QUOTAS.values()))

Target pilot size: 60


In [54]:
# Select a reproducible sample that represents ESG domains, document types and text lengths
def balanced_sample(group, sample_size, seed):
    # Return the complete group when it contains fewer records than the requested quota
    if len(group) <= sample_size:
        return group.copy()

    # Select one representative from every available domain–document–length combination
    strata = ["primary_esg_domain", "document_type", "length_band"]
    representatives = group.groupby(strata, observed=True, dropna=False, group_keys=False).sample(n=1, random_state=seed)

    # If the representatives exceed the quota, randomly retain the required number
    if len(representatives) >= sample_size:
        return representatives.sample(sample_size, random_state=seed)

    # Otherwise, fill the remaining places from candidates not already selected
    remaining = group[~group["evidence_id"].isin(representatives["evidence_id"])]
    additional = remaining.sample(sample_size - len(representatives), random_state=seed)
    return pd.concat([representatives, additional])

# Apply the specified quota independently to every structural evidence group
pilot_parts = []
for seed, (group_name, quota) in enumerate(PILOT_QUOTAS.items(), start=42):
    group = development_pool[development_pool["sampling_group"].eq(group_name)]
    selected = balanced_sample(group, min(quota, len(group)), seed)
    pilot_parts.append(selected)
    print(f"{group_name}: {len(selected)}/{quota}")

# Combine all structural groups into one development pilot
development_pilot_df = pd.concat(pilot_parts, ignore_index=True)

# Confirm that the pilot contains unique evidence and uses only development records
assert development_pilot_df["evidence_id"].is_unique
assert development_pilot_df["split"].eq("development").all()

print("Development pilot evidence:", len(development_pilot_df))

clause: 20/20
article: 12/12
point: 8/8
embedded_article: 8/8
oversized_unit: 4/4
inactive: 4/4
other: 4/4
Development pilot evidence: 60


In [55]:
# Confirm that the pilot is not dominated by one structure, domain or length
display(development_pilot_df["sampling_group"].value_counts().reindex(PILOT_QUOTAS).rename("selected").to_frame())
display(pd.crosstab(development_pilot_df["sampling_group"], development_pilot_df["length_band"]))
display(development_pilot_df["primary_esg_domain"].value_counts().rename("selected").to_frame())
display(development_pilot_df["document_type"].value_counts().rename("selected").to_frame())
print("Documents represented:", development_pilot_df["doc_id"].nunique())

,selected
sampling_group,
clause,20
article,12
point,8
embedded_article,8
oversized_unit,4
inactive,4
other,4


length_band,short,medium,long
sampling_group,,,
article,3,4,5
clause,10,7,3
embedded_article,1,3,4
inactive,3,1,0
other,0,2,2
oversized_unit,0,0,4
point,4,3,1


,selected
primary_esg_domain,
Environment,35
Governance,10
Unclassified,9
Social,6


,selected
document_type,
Circular,15
Law,13
Decree,10
Resolution,7
Decision,7
Ordinance,3
Integrated Document,3
Directive,1
Notification,1


Documents represented: 48


In [56]:
# Inspect metadata for all selected records and complete text for a reproducible subset
display(development_pilot_df[["evidence_id", "doc_id", "sampling_group", "evidence_type", "source_article", "primary_esg_domain", "document_type", "length_band", "text_length"]].sort_values(["sampling_group", "doc_id"]))

pilot_text_sample = development_pilot_df.groupby("sampling_group", group_keys=False).sample(n=1, random_state=42)
with pd.option_context("display.max_colwidth", None):
    display(pilot_text_sample[["doc_id", "sampling_group", "source_article", "source_text"]])

,evidence_id,doc_id,sampling_group,evidence_type,source_article,primary_esg_domain,document_type,length_band,text_length
20,05_2025_ND-CP_642709_article_0004,05_2025_ND-CP_642709,article,article,4,Environment,Decree,short,98
27,14_2023_TT-BNNPTNT_m_594179_article_0002,14_2023_TT-BNNPTNT_m_594179,article,article,2,Social,Circular,short,157
30,20_2009_TT-BXD_m_358651_article_0006,20_2009_TT-BXD_m_358651,article,article,6,Unclassified,Circular,medium,314
28,36_2016_TT-BTC_m_308617_article_0011,36_2016_TT-BTC_m_308617,article,article,11,Governance,Circular,long,4333
29,450_QD-TTg_m_510740_article_0002,450_QD-TTg_m_510740,article,article,2,Environment,Decision,long,2866
23,45_2019_QH14_432162_article_0103,45_2019_QH14_432162,article,article,103,Environment,Law,medium,276
26,66_19_2026_NQ-CP_707488_article_0001,66_19_2026_NQ-CP_707488,article,article,1,Environment,Resolution,medium,256
24,72_2020_QH14_m_463512_article_0061,72_2020_QH14_m_463512,article,article,61,Environment,Law,long,2407
21,73_2025_ND-CP_m_651068 (1)_article_0001,73_2025_ND-CP_m_651068 (1),article,article,1,Governance,Decree,medium,524
25,79_2023_ND-CP_m_590378_article_0031,79_2023_ND-CP_m_590378,article,article,31,Social,Decree,long,1308


doc_id    sampling_group source_article  \
30  20_2009_TT-BXD_m_358651           article              6   
15    73_VBHN-VPQH_m_707886            clause             62   
40     146_2025_QH15_706223  embedded_article              5   
53   07_2022_ND-CP_m_504176          inactive              1   
57      03_CT-NHNN_m_348086             other             II   
49  36_2016_TT-BTC_m_308617    oversized_unit             22   
37  34_2023_TT-BTC_m_568907             point              6   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

### **Validate pilot evidence** 

In [57]:
# Confirm that every selected passage exactly matches its saved source-document range
SPLIT_FOLDER = PROJECT_ROOT / "data" / "splits"
text_cache = {}

def validate_evidence_offset(row):
    path = SPLIT_FOLDER / row["split"] / row["cleaned_filename"]
    if path not in text_cache:
        text_cache[path] = path.read_text(encoding="utf-8", errors="replace")
    extracted = text_cache[path][int(row["evidence_start_char"]):int(row["evidence_end_char"])].strip()
    return extracted == str(row["source_text"]).strip()

development_pilot_df["offset_valid"] = development_pilot_df.apply(validate_evidence_offset, axis=1)
assert development_pilot_df["offset_valid"].all(), "Pilot evidence contains offset mismatches."
print("Pilot offset validation passed:", len(development_pilot_df))

Pilot offset validation passed: 60


In [58]:
# Add manual-review fields without automatically approving or rejecting evidence
review_df = development_pilot_df.copy()
review_df["review_status"] = "pending"
review_df["review_notes"] = ""
review_df["question_type_hint"] = ""
review_df["context_suggestion"] = ""

# Flag short subordinate provisions that may require their parent Clause or Article
context_mask = review_df["evidence_type"].isin(["point", "lettered_item", "numbered_item"]) & review_df["text_length"].lt(100)
review_df.loc[context_mask, "context_suggestion"] = "Check whether parent context is required"

review_columns = ["evidence_id", "doc_id", "sampling_group", "document_type", "primary_esg_domain", "parent_article_number", "evidence_type", "evidence_number", "evidence_title", "text_length", "offset_valid", "context_suggestion", "source_text", "review_status", "review_notes", "question_type_hint"]
review_df = review_df[review_columns].sort_values(["sampling_group", "doc_id"]).reset_index(drop=True)

print("Evidence awaiting review:", review_df["review_status"].eq("pending").sum())
display(review_df.drop(columns="source_text"))

Evidence awaiting review: 60


,evidence_id,doc_id,sampling_group,document_type,primary_esg_domain,parent_article_number,evidence_type,evidence_number,evidence_title,text_length,offset_valid,context_suggestion,review_status,review_notes,question_type_hint
0,05_2025_ND-CP_642709_article_0004,05_2025_ND-CP_642709,article,Decree,Environment,4,article,4,Implementation clause,98,True,,pending,,
1,14_2023_TT-BNNPTNT_m_594179_article_0002,14_2023_TT-BNNPTNT_m_594179,article,Circular,Social,2,article,2,Importation of animal breeds and aquatic breed...,157,True,,pending,,
2,20_2009_TT-BXD_m_358651_article_0006,20_2009_TT-BXD_m_358651,article,Circular,Unclassified,6,article,6,Section II Part III shall be amended as follows:,314,True,,pending,,
3,36_2016_TT-BTC_m_308617_article_0011,36_2016_TT-BTC_m_308617,article,Circular,Governance,11,article,11,Finalization of natural resources tax,4333,True,,pending,,
4,450_QD-TTg_m_510740_article_0002,450_QD-TTg_m_510740,article,Decision,Environment,2,article,2,"Plans, resources, and organization of the Stra...",2866,True,,pending,,
5,45_2019_QH14_432162_article_0103,45_2019_QH14_432162,article,Law,Environment,103,article,103,Pay rise,276,True,,pending,,
6,66_19_2026_NQ-CP_707488_article_0001,66_19_2026_NQ-CP_707488,article,Resolution,Environment,1,article,1,Scope,256,True,,pending,,
7,72_2020_QH14_m_463512_article_0061,72_2020_QH14_m_463512,article,Law,Environment,61,article,61,Environmental protection in agricultural produ...,2407,True,,pending,,
8,73_2025_ND-CP_m_651068 (1)_article_0001,73_2025_ND-CP_m_651068 (1),article,Decree,Governance,1,article,1,Amendment of preferential import tariff rates ...,524,True,,pending,,
9,79_2023_ND-CP_m_590378_article_0031,79_2023_ND-CP_m_590378,article,Decree,Social,31,article,31,Recognition of plant variety right representat...,1308,True,,pending,,


In [59]:
# Save a persistent review sheet for manual evidence approval
QA_FOLDER = PROJECT_ROOT / "outputs" / "qa_benchmark"
QA_FOLDER.mkdir(parents=True, exist_ok=True)
REVIEW_PATH = QA_FOLDER / "development_evidence_review.csv"
review_df.to_csv(REVIEW_PATH, index=False, encoding="utf-8-sig")

print("Review sheet saved:", REVIEW_PATH)

Review sheet saved: /Users/tanggiee/Desktop/RAG_AI/esg_rag_project/outputs/qa_benchmark/development_evidence_review.csv


### **Context enrichment** 

In [60]:
# Find the existing review CSV without creating or changing anything
review_matches = list(PROJECT_ROOT.rglob("development_evidence_review.csv"))
print("Matches found:", review_matches)
assert len(review_matches) == 1, f"Expected one review CSV, found {len(review_matches)}"

REVIEW_PATH = review_matches[0]
BENCHMARK_FOLDER = REVIEW_PATH.parent
print("Review file:", REVIEW_PATH)

Matches found: [PosixPath('/Users/tanggiee/Desktop/RAG_AI/esg_rag_project/outputs/qa_benchmark/development_evidence_review.csv')]
Review file: /Users/tanggiee/Desktop/RAG_AI/esg_rag_project/outputs/qa_benchmark/development_evidence_review.csv


In [62]:
# Load the existing review CSV and automatically detect its delimiter
BENCHMARK_FOLDER = PROJECT_ROOT / "outputs" / "qa_benchmark"
REVIEW_PATH = BENCHMARK_FOLDER / "development_evidence_review.csv"
assert REVIEW_PATH.exists(), f"Review file not found: {REVIEW_PATH}"

review_df = pd.read_csv(REVIEW_PATH, sep=None, engine="python", encoding="utf-8-sig")
review_df["review_status"] = review_df["review_status"].str.strip().str.lower()

# Attach manual decisions to the original parser-backed evidence records
evidence_base_df = evidence_candidates_df.drop(columns=["review_status", "review_notes"], errors="ignore")
review_labels = review_df[["evidence_id", "review_status", "review_notes"]]
reviewed_pilot_df = evidence_base_df.merge(review_labels, on="evidence_id", how="inner", validate="one_to_one")

assert len(reviewed_pilot_df) == 60
assert reviewed_pilot_df["evidence_id"].is_unique
assert set(reviewed_pilot_df["review_status"]) <= {"approved", "needs_context", "rejected"}

display(reviewed_pilot_df["review_status"].value_counts().rename("record_count").to_frame())

,record_count
review_status,
approved,51
needs_context,8
rejected,1


In [63]:
# Retain approved and context-dependent evidence; exclude rejected records
qa_evidence_df = reviewed_pilot_df[~reviewed_pilot_df["review_status"].eq("rejected")].copy()
qa_evidence_df["original_evidence_text"] = qa_evidence_df["source_text"]
qa_evidence_df["context_text"] = qa_evidence_df["source_text"]
qa_evidence_df["context_source_id"] = qa_evidence_df["evidence_id"]
qa_evidence_df["context_level"] = qa_evidence_df["evidence_level"]
qa_evidence_df["context_added"] = False

# Prepare parent-Article and Clause lookups
units_lookup = units_df.set_index("unit_id")
clauses_df = provisions_df[provisions_df["provision_type"].eq("clause")].copy()
clauses_df["context_length"] = clauses_df["end_char"] - clauses_df["start_char"]

# Add context only to evidence marked needs_context
for index in qa_evidence_df.index[qa_evidence_df["review_status"].eq("needs_context")]:
    row = qa_evidence_df.loc[index]
    containing_clauses = clauses_df[(clauses_df["parent_unit_id"].eq(row.get("parent_unit_id"))) & clauses_df["start_char"].le(row["evidence_start_char"]) & clauses_df["end_char"].ge(row["evidence_end_char"])]

    # Prefer the smallest Clause containing a subordinate Point
    if row["evidence_type"] in {"point", "numbered_item", "lettered_item"} and not containing_clauses.empty:
        context = containing_clauses.sort_values("context_length").iloc[0]
        qa_evidence_df.loc[index, ["context_text", "context_source_id", "context_level", "context_added"]] = [context["provision_text"], context["provision_id"], "clause", True]
    else:
        # Clauses and Points without a containing Clause receive their parent Article
        parent_unit_id = row.get("parent_unit_id") if pd.notna(row.get("parent_unit_id")) else row["evidence_id"]
        parent = units_lookup.loc[parent_unit_id]
        qa_evidence_df.loc[index, ["context_text", "context_source_id", "context_level", "context_added"]] = [parent["unit_text"], parent_unit_id, parent["unit_type"], True]

# Use the enriched context as the QA-generation source while preserving original evidence
qa_evidence_df["qa_source_text"] = qa_evidence_df["context_text"]

assert len(qa_evidence_df) == 59
assert qa_evidence_df["evidence_id"].is_unique
assert qa_evidence_df["qa_source_text"].notna().all()
assert qa_evidence_df["context_added"].sum() == 8

print("Final pilot records:", len(qa_evidence_df))
print("Context added:", qa_evidence_df["context_added"].sum())
display(pd.crosstab(qa_evidence_df["review_status"], qa_evidence_df["context_level"]))

Final pilot records: 59
Context added: 8


context_level,article,clause,provision,unit
review_status,,,,
approved,0,0,32,19
needs_context,4,4,0,0


In [64]:
# Compare the original evidence with the context added after manual review
context_review_df = qa_evidence_df[qa_evidence_df["context_added"]][["evidence_id", "evidence_type", "parent_article_number", "original_evidence_text", "context_level", "context_source_id", "context_text"]]

with pd.option_context("display.max_colwidth", None):
    display(context_review_df)

evidence_id evidence_type  \
25      41_2024_QH15_m_622824_article_0020_point_001         point   
28     81_2023_QH15_m_565003_article_0012_clause_005        clause   
35    125_VBHN-VPQH_m_682303_article_0015_clause_001        clause   
46    203_2025_QH15_m_661982_article_0001_clause_001        clause   
49     07_2022_ND-CP_m_504176_article_0001_point_099         point   
52   37_2011_QD-TTg_m_126596_article_0015_clause_001        clause   
56  10_2014_UBTVQH13_m_267874_article_0008_point_002         point   
59   19_2009_TT-BKHCN_m_96067_article_0002_point_011         point   

   parent_article_number  \
25                    20   
28                    12   
35                    15   
46                     1   
49                     1   
52                    15   
56                     8   
59                     2   

                                                                                                                                                                   original_evidence_text  \
25                                                                  a) Schemes, plans for preservation and growth of social insurance, unemployment insurance and health insurance funds.   
28  5. Expanding land of industrial parks, especially in dynamic regions, connected with economic corridors. Continuing to expand urban land associated with the process of urbanization.   
35                                                                                                                       1. Voluntariness, equality, good faith, cooperation and honesty.   
46                                                                                                                                                    1. Article 9 is amended as follows:   
49                                                                                                                                                      d) Point g Clause 5 is abrogated.   
52                                                      1. The grid-unlinked wind power project is enjoyed preferential, supporting specified in article 12, article 13 of this Decision.   
56                                                                                                       b) Environmental Police Divisions affiliated to police authorities of provinces;   
59                                                                                                                                                    h/ Warnings on safely for laborers.   

   context_level                                  context_source_id  \
25        clause      41_2024_QH15_m_622824_article_0020_clause_004   
28       article                 81_2023_QH15_m_565003_article_0012   
35       article                125_VBHN-VPQH_m_682303_article_0015   
46       article                203_2025_QH15_m_661982_article_0001   
49        clause     07_2022_ND-CP_m_504176_article_0001_clause_033   
52       article               37_2011_QD-TTg_m_126596_article_0015   
56        clause  10_2014_UBTVQH13_m_267874_article_0008_clause_001   
59        clause   19_2009_TT-BKHCN_m_96067_article_0002_clause_003   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [65]:
# Final checks before saving the context-enriched QA evidence
assert len(qa_evidence_df) == 59
assert qa_evidence_df["evidence_id"].is_unique
assert qa_evidence_df["split"].eq("development").all()
assert qa_evidence_df["original_evidence_text"].notna().all()
assert qa_evidence_df["qa_source_text"].notna().all()
assert not qa_evidence_df["review_status"].eq("rejected").any()
assert qa_evidence_df["review_status"].eq("needs_context").sum() == qa_evidence_df["context_added"].sum() == 8

# Save JSONL for QA generation and CSV for convenient inspection
QA_INPUT_PATH = BENCHMARK_FOLDER / "development_qa_evidence.jsonl"
QA_AUDIT_PATH = BENCHMARK_FOLDER / "development_qa_evidence.csv"
qa_evidence_df.to_json(QA_INPUT_PATH, orient="records", lines=True, force_ascii=False)
qa_evidence_df.to_csv(QA_AUDIT_PATH, index=False, encoding="utf-8-sig")

print("QA evidence records:", len(qa_evidence_df))
print("Direct evidence:", (~qa_evidence_df["context_added"]).sum())
print("Context-enriched evidence:", qa_evidence_df["context_added"].sum())
print("JSONL:", QA_INPUT_PATH)
print("Audit CSV:", QA_AUDIT_PATH)

QA evidence records: 59
Direct evidence: 51
Context-enriched evidence: 8
JSONL: /Users/tanggiee/Desktop/RAG_AI/esg_rag_project/outputs/qa_benchmark/development_qa_evidence.jsonl
Audit CSV: /Users/tanggiee/Desktop/RAG_AI/esg_rag_project/outputs/qa_benchmark/development_qa_evidence.csv


In [66]:
# Define QA schema 

In [67]:
# Regulatory categories retained in the benchmark
QUESTION_TYPES = ["definition", "scope", "applicability", "obligation", "prohibition", "procedure", "deadline", "condition", "exception", "authority", "legal_status", "other"]
CORE_COMPLIANCE_TYPES = {"obligation", "prohibition", "procedure", "deadline", "condition", "exception"}
SUPPORTING_REGULATORY_TYPES = {"definition", "scope", "applicability", "authority", "legal_status", "other"}
REJECTION_REASONS = ["insufficient_context", "non_substantive", "ambiguous", "not_regulatory", "duplicate", "parser_error", "other"]

QuestionType = Literal["", "definition", "scope", "applicability", "obligation", "prohibition", "procedure", "deadline", "condition", "exception", "authority", "legal_status", "other"]
ComplianceGroup = Literal["", "core_compliance", "supporting_regulatory"]
RejectionReason = Literal["", "insufficient_context", "non_substantive", "ambiguous", "not_regulatory", "duplicate", "parser_error", "other"]

class SupportingExcerpt(BaseModel):
    model_config = ConfigDict(extra="forbid")
    evidence_id: str
    excerpt: str

class GeneratedQA(BaseModel):
    model_config = ConfigDict(extra="forbid")
    usable: bool
    question_scope: Literal["", "single_passage"]
    compliance_group: ComplianceGroup
    question_type: QuestionType
    question: str
    answer: str
    supporting_excerpts: list[SupportingExcerpt]
    rejection_reason: RejectionReason

In [68]:
# Convert missing metadata into empty strings instead of "nan"
def clean_value(value):
    return "" if pd.isna(value) else str(value).strip()

# Build one novice-friendly, evidence-grounded regulatory QA prompt
def build_qa_prompt(row):
    return f"""You are generating evidence-grounded question-answer pairs for evaluating an ESG regulatory-compliance retrieval system.

USER PERSPECTIVE

Assume the user has limited knowledge of the regulation. The user may not know the document name, Article number, legal terminology, responsible authority or exact compliance requirement.

Generate the type of practical question that a compliance officer, business representative, employee, investor or member of the public might ask when trying to understand what a regulation requires.

TASK

Generate exactly one realistic regulatory question and answer from the approved evidence.

Prioritize substantive compliance requirements concerning obligations, prohibitions, procedures, deadlines, conditions and exceptions. Definitions, scope, applicability, authority and legal status may be used when they contain information that is necessary for understanding or applying a regulation.

QUESTION RULES

1. Make the question standalone and understandable without displaying the evidence or metadata.
2. Use clear language suitable for a user with limited regulatory knowledge.
3. Ask a realistic regulatory-guidance question, not a reading-comprehension question.
4. Identify the regulated person, organization, activity, product, authority or situation when stated in the evidence.
5. Do not require knowledge of an Article or Clause number unless it is necessary to distinguish the rule.
6. Prefer formulations such as “What must...?”, “Who is required to...?”, “What steps...?”, “How long...?”, “Under what conditions...?” or “When does this requirement not apply?”
7. Avoid vague questions such as “What does the law say?” or “What are the requirements?”
8. Avoid questions that merely repeat a heading.
9. Avoid yes/no questions when a more informative question is possible.
10. Do not invent a company, scenario, authority, deadline, consequence or interpretation.

GROUNDING RULES

1. Use only APPROVED QA EVIDENCE.
2. Do not use external knowledge, assumptions or invented details.
3. Every material answer statement must be supported by APPROVED QA EVIDENCE.
4. Preserve controlling qualifications, exceptions, thresholds, deadlines, responsible parties and procedural alternatives.
5. Copy one exact and continuous supporting excerpt from APPROVED QA EVIDENCE.
6. Use the evidence identifier exactly as supplied.
7. Do not insert metadata into the answer unless it also appears in the evidence.
8. Reject the record instead of inventing missing context.

CLASSIFICATION

Assign exactly one question_type from:
{", ".join(QUESTION_TYPES)}

Assign compliance_group as:
- core_compliance for obligation, prohibition, procedure, deadline, condition or exception;
- supporting_regulatory for definition, scope, applicability, authority, legal_status or other.

REJECTION RULES

If the evidence cannot support a clear and substantive regulatory question:
- set usable to false;
- set question_scope, compliance_group, question_type, question and answer to empty strings;
- set supporting_excerpts to an empty list;
- select rejection_reason from: {", ".join(REJECTION_REASONS)}.

If the evidence is usable:
- set usable to true;
- set question_scope to single_passage;
- set rejection_reason to an empty string;
- return exactly one supporting excerpt.

OUTPUT RULES

Return only one valid JSON object matching the required schema. Do not include Markdown, explanations or additional fields.

METADATA

Evidence ID: {clean_value(row.get("evidence_id"))}
Document type: {clean_value(row.get("document_type"))}
ESG domain: {clean_value(row.get("primary_esg_domain"))}
Parent Article: {clean_value(row.get("parent_article_number"))}
Evidence type: {clean_value(row.get("evidence_type"))}
Evidence number: {clean_value(row.get("evidence_number"))}
Evidence title: {clean_value(row.get("evidence_title"))}

APPROVED QA EVIDENCE

{clean_value(row.get("qa_source_text"))}

ORIGINAL GOLD EXTRACT

{clean_value(row.get("original_evidence_text"))}
"""

### **Generate QA test** 

In [69]:
import json, os, time
from datetime import datetime, timezone
from getpass import getpass
from openai import OpenAI

In [70]:
# New prompt version creates a separate checkpoint and preserves v1
QA_MODEL = "gpt-5.6"
QA_PROMPT_VERSION = "2.3.5-v2"
MAX_RECORDS = 3  # Keep 3 for initial validation; change to None after approval
GENERATION_PATH = BENCHMARK_FOLDER / f"development_qa_generated_{QA_PROMPT_VERSION}.jsonl"

# Request the key privately when the environment does not already contain it
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
client = OpenAI()

In [71]:
# Reject malformed, inconsistent or ungrounded model output
def validate_generated_qa(result, row):
    approved_text = clean_value(row["qa_source_text"])
    if result.usable:
        assert result.question_scope == "single_passage"
        assert result.question_type in QUESTION_TYPES
        expected_group = "core_compliance" if result.question_type in CORE_COMPLIANCE_TYPES else "supporting_regulatory"
        assert result.compliance_group == expected_group
        assert result.question.strip() and result.answer.strip()
        assert len(result.supporting_excerpts) == 1
        assert result.supporting_excerpts[0].evidence_id == row["evidence_id"]
        assert result.supporting_excerpts[0].excerpt.strip()
        assert result.supporting_excerpts[0].excerpt in approved_text
        assert result.rejection_reason == ""
    else:
        assert result.question_scope == result.compliance_group == result.question_type == ""
        assert result.question == result.answer == ""
        assert result.supporting_excerpts == []
        assert result.rejection_reason in REJECTION_REASONS

# Read completed checkpoint records so reruns skip successful evidence IDs
def load_generation_checkpoint(path):
    if not path.exists():
        return []
    records = []
    with path.open(encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if line.strip():
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError as error:
                    raise ValueError(f"Invalid JSON on checkpoint line {line_number}: {error}")
    return records

completed_records = load_generation_checkpoint(GENERATION_PATH)
completed_ids = {record["evidence_id"] for record in completed_records}
assert len(completed_ids) == len(completed_records), "Duplicate evidence IDs in checkpoint"

# Use the same evidence records as v1 so prompt versions can be compared fairly
pending_df = qa_evidence_df[~qa_evidence_df["evidence_id"].isin(completed_ids)].copy()
if MAX_RECORDS is not None:
    pending_df = pending_df.head(MAX_RECORDS)

print("Previously completed:", len(completed_ids))
print("Generating now:", len(pending_df))

# Generate, validate and immediately save each successful result
for number, (_, row) in enumerate(pending_df.iterrows(), start=1):
    try:
        response = client.responses.parse(
            model=QA_MODEL,
            input=[
                {"role": "system", "content": "Generate strictly evidence-grounded regulatory QA data using the required schema."},
                {"role": "user", "content": build_qa_prompt(row)}
            ],
            text_format=GeneratedQA
        )
        result = response.output_parsed
        assert result is not None, "No parsed output returned"
        validate_generated_qa(result, row)

        record = {
            "qa_id": f"{row['evidence_id']}_q01",
            "evidence_id": row["evidence_id"],
            "doc_id": row["doc_id"],
            "model": QA_MODEL,
            "prompt_version": QA_PROMPT_VERSION,
            "response_id": response.id,
            "generated_at_utc": datetime.now(timezone.utc).isoformat(),
            **result.model_dump()
        }

        with GENERATION_PATH.open("a", encoding="utf-8") as file:
            file.write(json.dumps(record, ensure_ascii=False) + "\n")

        print(f"[{number}/{len(pending_df)}] saved | {row['evidence_id']} | {result.compliance_group} | usable={result.usable}")
        time.sleep(0.5)

    except Exception as error:
        error_text = str(error)
        print(f"[{number}/{len(pending_df)}] ERROR | {row['evidence_id']} | {type(error).__name__}: {error}")

        # Stop only when API credit is unavailable; successful records remain saved
        if "credit_balance_exhausted" in error_text or "insufficient_quota" in error_text:
            print("Generation stopped. Add API credits and rerun this cell.")
            break

        # Wait briefly after temporary request-limit errors
        if "429" in error_text or "rate limit" in error_text.lower():
            time.sleep(15)

# Reload and inspect the complete v2 checkpoint
generated_records = load_generation_checkpoint(GENERATION_PATH)
generated_df = pd.DataFrame(generated_records)

print("\nCheckpoint:", GENERATION_PATH)
print("Completed v2 records:", len(generated_df))
if not generated_df.empty:
    display(generated_df[["evidence_id", "usable", "question_scope", "compliance_group", "question_type", "question", "answer", "supporting_excerpts", "rejection_reason"]].tail(max(3, len(pending_df))))

Previously completed: 60
Generating now: 0

Checkpoint: /Users/tanggiee/Desktop/RAG_AI/esg_rag_project/outputs/qa_benchmark/development_qa_generated_2.3.5-v2.jsonl
Completed v2 records: 60


,evidence_id,usable,question_scope,compliance_group,question_type,question,answer,supporting_excerpts,rejection_reason
57,10_2014_UBTVQH13_m_267874_article_0011_clause_002,True,single_passage,core_compliance,obligation,What protections must be provided to organizat...,Every organization and individual that coopera...,[{'evidence_id': '10_2014_UBTVQH13_m_267874_ar...,
58,04_2026_QD-TTg_m_695772_article_0012_clause_001,True,single_passage,core_compliance,procedure,How may organizations and individuals submit a...,They may submit the application to the field-s...,[{'evidence_id': '04_2026_QD-TTg_m_695772_arti...,
59,19_2009_TT-BKHCN_m_96067_article_0002_point_011,False,,,,,,[],insufficient_context


In [72]:
# Display the complete generated QA text without truncating long cells
display_columns = ["evidence_id", "usable", "question_scope", "compliance_group", "question_type", "question", "answer", "supporting_excerpts", "rejection_reason"]
rows_to_show = generated_df.tail(max(3, len(pending_df)))

with pd.option_context("display.max_colwidth", None, "display.max_columns", None, "display.width", None, "display.max_rows", None):
    display(rows_to_show[display_columns])

,evidence_id,usable,question_scope,compliance_group,question_type,question,answer,supporting_excerpts,rejection_reason
57,10_2014_UBTVQH13_m_267874_article_0011_clause_002,True,single_passage,core_compliance,obligation,What protections must be provided to organizations and individuals who cooperate with environmental police forces?,Every organization and individual that cooperates with environmental police forces must be physically protected and have their confidentiality protected in accordance with the related laws.,"[{'evidence_id': '10_2014_UBTVQH13_m_267874_article_0011_clause_002', 'excerpt': '2. Every organization and individual that cooperates with environmental police forces shall be physically and confidentiality protected in accordance with the related laws.'}]",
58,04_2026_QD-TTg_m_695772_article_0012_clause_001,True,single_passage,core_compliance,procedure,"How may organizations and individuals submit an application for inspection and approval of an oil spill response plan, and what is the receipt date if they submit it by post?","They may submit the application to the field-specific authority responsible for inspection and approval through the National document exchange platform, an online public service portal, or a postal service. For postal submissions, the receipt date is the date on which the postal service delivers the application to the field-specific authority.","[{'evidence_id': '04_2026_QD-TTg_m_695772_article_0012_clause_001', 'excerpt': 'Organizations and individuals submitting application for inspection and approval for plans for oil spill response via the National document exchange platform or via online public service portal or via post service to field-specific authorities tasked with inspection and approval in order to receive the application; where application is submitted via post service, the date on which field-specific authorities receive the application shall be the date on which post service delivers the application to the field-specific authorities.'}]",
59,19_2009_TT-BKHCN_m_96067_article_0002_point_011,False,,,,,,[],insufficient_context


In [73]:
# 2.3.6. Generate all remaining development QA records using the validated v2 configuration
# Load the checkpoint and keep one successful generation per evidence record
completed_records = load_generation_checkpoint(GENERATION_PATH)
completed_df = pd.DataFrame(completed_records).drop_duplicates("evidence_id", keep="first") if completed_records else pd.DataFrame()
completed_ids = set(completed_df["evidence_id"]) if not completed_df.empty else set()

# Rewrite the checkpoint only when duplicate records were found
if len(completed_records) != len(completed_ids):
    with GENERATION_PATH.open("w", encoding="utf-8") as file:
        for record in completed_df.to_dict("records"):
            file.write(json.dumps(record, ensure_ascii=False) + "\n")

# Select only unfinished development evidence; completed records will not be charged or generated again
pending_df = qa_evidence_df[~qa_evidence_df["evidence_id"].isin(completed_ids)].copy()
print("Previously completed:", len(completed_ids))
print("Remaining records:", len(pending_df))

Previously completed: 60
Remaining records: 0


In [74]:
# Generate one QA pair per remaining evidence record and checkpoint every successful response
for number, (_, row) in enumerate(pending_df.iterrows(), start=1):
    try:
        response = client.responses.parse(model=QA_MODEL, input=[{"role": "system", "content": "Generate strictly evidence-grounded regulatory compliance QA data using the required schema."}, {"role": "user", "content": build_qa_prompt(row)}], text_format=GeneratedQA)
        result = response.output_parsed
        assert result is not None, "No structured output was returned"
        validate_generated_qa(result, row)

        # Attach identifiers and provenance from the source data rather than asking the model to generate them
        record = {"qa_id": f"{row['evidence_id']}_q01", "evidence_id": row["evidence_id"], "doc_id": row["doc_id"], "model": QA_MODEL, "prompt_version": QA_PROMPT_VERSION, "response_id": response.id, "generated_at_utc": datetime.now(timezone.utc).isoformat(), **result.model_dump()}

        with GENERATION_PATH.open("a", encoding="utf-8") as file:
            file.write(json.dumps(record, ensure_ascii=False) + "\n")

        completed_ids.add(row["evidence_id"])
        print(f"[{number}/{len(pending_df)}] saved | {row['evidence_id']} | {result.compliance_group} | usable={result.usable}")
        time.sleep(0.3)

    except Exception as error:
        error_text = str(error)
        print(f"[{number}/{len(pending_df)}] ERROR | {row['evidence_id']} | {type(error).__name__}: {error}")

        # Stop only when credits are unavailable; rerunning later safely resumes from the checkpoint
        if "credit_balance_exhausted" in error_text or "insufficient_quota" in error_text:
            print("Generation stopped because API credits are unavailable.")
            break

In [75]:
# Reload the complete checkpoint and confirm that each evidence record appears only once
generated_records = load_generation_checkpoint(GENERATION_PATH)
generated_df = pd.DataFrame(generated_records).drop_duplicates("evidence_id", keep="first")
remaining_ids = set(qa_evidence_df["evidence_id"]) - set(generated_df["evidence_id"])

AssertionError: 

In [76]:
# Export a readable CSV copy for complete development-set review
GENERATION_CSV_PATH = GENERATION_PATH.with_suffix(".csv")
generated_df.to_csv(GENERATION_CSV_PATH, index=False, encoding="utf-8-sig")

print("\nSuccessful unique records:", len(generated_df))
print("Records still unfinished:", len(remaining_ids))
print("Usable:", int(generated_df["usable"].sum()))
print("Rejected:", int((~generated_df["usable"]).sum()))
print("Checkpoint:", GENERATION_PATH)
print("Review CSV:", GENERATION_CSV_PATH)

if remaining_ids:
    display(pd.DataFrame({"unfinished_evidence_id": sorted(remaining_ids)}))
else:
    print("Development QA generation is complete.")


Successful unique records: 60
Records still unfinished: 0
Usable: 51
Rejected: 9
Checkpoint: /Users/tanggiee/Desktop/RAG_AI/esg_rag_project/outputs/qa_benchmark/development_qa_generated_2.3.5-v2.jsonl
Review CSV: /Users/tanggiee/Desktop/RAG_AI/esg_rag_project/outputs/qa_benchmark/development_qa_generated_2.3.5-v2.csv
Development QA generation is complete.


#### **create another QA dataset for review / comparison**

In [77]:
# define direct ESG content rules to create directly relevant QA pairs 
# High-precision terms used only to create an ESG candidate pool, final inclusion is manually confirmed
import re

ESG_PATTERNS = {
    "E": re.compile(r"\b(environmental protection|environmental impact|climate change|greenhouse gas|carbon emission|air pollution|water pollution|wastewater|hazardous waste|solid waste|recycling|circular economy|renewable energy|energy efficiency|biodiversity|ecosystem|forest protection|deforestation|land degradation|soil pollution|natural resources|mineral extraction|sustainable agriculture)\b", re.I),
    "S": re.compile(r"\b(labou?r rights?|employees?|workers?|employment|occupational safety|workplace safety|social insurance|social security|minimum wage|working hours|discrimination|gender equality|disabled persons?|elderly persons?|child labou?r|human rights?|public health|community health|resettlement|indigenous|social housing)\b", re.I),
    "G": re.compile(r"\b(corporate governance|board of directors|shareholders?|audit committee|internal control|risk management|regulatory disclosure|sustainability report|environmental disclosure|anti-corruption|corruption|bribery|conflict of interest|business ethics|accountability|corporate transparency|beneficial owner|whistleblow|data protection|cybersecurity)\b", re.I)
}

# Remove agency names that could create false Environmental matches
AGENCY_PATTERN = re.compile(r"\b(Ministry|Department) of (Agriculture and Environment|Natural Resources and Environment)\b", re.I)

def detect_direct_esg(text):
    clean_text = AGENCY_PATTERN.sub("", str(text or ""))
    pillars = [pillar for pillar, pattern in ESG_PATTERNS.items() if pattern.search(clean_text)]
    return ",".join(pillars)

evidence_candidates_df["detected_esg_pillars"] = evidence_candidates_df["source_text"].fillna("").apply(detect_direct_esg)
evidence_candidates_df["direct_esg_candidate"] = evidence_candidates_df["detected_esg_pillars"].ne("")

display(evidence_candidates_df.groupby(["split", "detected_esg_pillars"]).size().rename("candidate_count").to_frame())

candidate_count
split       detected_esg_pillars                 
development                                 58821
            E                                7726
            E,G                                65
            E,S                               257
            E,S,G                              23
            G                                 646
            S                                4845
            S,G                                40
eval                                         9434
            E                                 716
            E,G                                 3
            E,S                                20
            E,S,G                               1
            G                                 309
            S                                 807
            S,G                                17
test                                        13274
            E                                 729
            E,G                                 5
            E,S                                40
            E,S,G                               4
            G                                 638
            S                                 243
            S,G                                43

In [78]:
# create a new ESG-focused review sample 

# Select development evidence only and exclude automatically unsuitable records
ESG_REVIEW_SIZE = 80
development_esg_pool = evidence_candidates_df[evidence_candidates_df["split"].eq("development") & evidence_candidates_df["direct_esg_candidate"] & evidence_candidates_df["eligible"]].copy()
development_esg_pool["text_length"] = development_esg_pool["source_text"].fillna("").str.len()
development_esg_pool["length_band"] = pd.cut(development_esg_pool["text_length"], bins=[0, 200, 800, 2500, float("inf")], labels=["short", "medium", "long", "very_long"])

# Sample across ESG pillar, evidence type, document type and evidence length
strata = ["detected_esg_pillars", "evidence_type", "document_type", "length_band"]
representatives = development_esg_pool.groupby(strata, observed=True, dropna=False, group_keys=False).sample(n=1, random_state=42)
if len(representatives) >= ESG_REVIEW_SIZE:     
    esg_review_df = representatives.sample(ESG_REVIEW_SIZE, random_state=42)
else:
    remaining = development_esg_pool[~development_esg_pool["evidence_id"].isin(representatives["evidence_id"])]
    esg_review_df = pd.concat([representatives, remaining.sample(min(ESG_REVIEW_SIZE - len(representatives), len(remaining)), random_state=42)])

# Add manual-review fields; review the passage itself, not merely its parent document label
esg_review_df = esg_review_df.drop_duplicates("evidence_id").reset_index(drop=True)
esg_review_df["esg_relevance"] = "pending"
esg_review_df["final_esg_pillar"] = ""
esg_review_df["esg_review_notes"] = ""

ESG_REVIEW_PATH = BENCHMARK_FOLDER / "development_direct_esg_review.csv"
review_columns = ["evidence_id", "doc_id", "document_type", "evidence_type", "evidence_number", "evidence_title", "detected_esg_pillars", "source_text", "esg_relevance", "final_esg_pillar", "esg_review_notes"]
esg_review_df[review_columns].to_csv(ESG_REVIEW_PATH, index=False, encoding="utf-8-sig")

print("Direct-ESG candidates available:", len(development_esg_pool))
print("Evidence records selected for review:", len(esg_review_df))
print("Review CSV:", ESG_REVIEW_PATH)

Direct-ESG candidates available: 13577
Evidence records selected for review: 80
Review CSV: /Users/tanggiee/Desktop/RAG_AI/esg_rag_project/outputs/qa_benchmark/development_direct_esg_review.csv


In [80]:
# Load manually confirmed passage-level ESG decisions
review_df = pd.read_csv(ESG_REVIEW_PATH, sep=None, engine="python", encoding="utf-8-sig")
review_df["esg_relevance"] = review_df["esg_relevance"].astype(str).str.strip().str.lower()
review_df["final_esg_pillar"] = review_df["final_esg_pillar"].astype(str).str.strip().str.upper()

assert set(review_df["esg_relevance"]) <= {"direct", "not_direct"}, "Complete every ESG relevance decision first"
assert set(review_df.loc[review_df["esg_relevance"].eq("direct"), "final_esg_pillar"]) <= {"E", "S", "G", "MULTI"}

direct_ids = review_df.loc[review_df["esg_relevance"].eq("direct"), "evidence_id"]
direct_esg_df = evidence_candidates_df[evidence_candidates_df["evidence_id"].isin(direct_ids)].merge(review_df[["evidence_id", "final_esg_pillar", "esg_review_notes"]], on="evidence_id", how="inner", validate="one_to_one")

assert len(direct_esg_df) >= 60, f"Only {len(direct_esg_df)} direct ESG records were approved; review additional candidates before continuing"

# Select 60 records while retaining diversity across ESG pillar, evidence type and document type
strata = ["final_esg_pillar", "evidence_type", "document_type"]
representatives = direct_esg_df.groupby(strata, observed=True, dropna=False, group_keys=False).sample(n=1, random_state=43)
if len(representatives) >= 60:
    qa_evidence_df = representatives.sample(60, random_state=43)
else:
    remaining = direct_esg_df[~direct_esg_df["evidence_id"].isin(representatives["evidence_id"])]
    qa_evidence_df = pd.concat([representatives, remaining.sample(60 - len(representatives), random_state=43)])

qa_evidence_df = qa_evidence_df.drop_duplicates("evidence_id").reset_index(drop=True)
# Map the reviewed passage into the text fields used by the existing QA prompt and validator
qa_evidence_df["qa_source_text"] = qa_evidence_df["source_text"]
qa_evidence_df["original_evidence_text"] = qa_evidence_df["source_text"]
assert len(qa_evidence_df) == 60
assert qa_evidence_df["evidence_id"].is_unique
assert qa_evidence_df["split"].eq("development").all()

display(qa_evidence_df.groupby(["final_esg_pillar", "evidence_type"]).size().unstack(fill_value=0))

evidence_type,article,clause,embedded_article,lettered_item,numbered_item,point,roman_section
final_esg_pillar,,,,,,,
E,4,4,4,2,0,5,2
G,2,3,1,1,1,0,0
MULTI,5,9,1,1,0,5,0
S,4,3,1,0,0,2,0


In [81]:
required_names = [
    "build_qa_prompt",
    "GeneratedQA",
    "validate_generated_qa",
    "QUESTION_TYPES",
    "CORE_COMPLIANCE_TYPES",
    "REJECTION_REASONS"
]

for name in required_names:
    print(name, "available" if name in globals() else "MISSING")

build_qa_prompt available
GeneratedQA available
validate_generated_qa available
QUESTION_TYPES available
CORE_COMPLIANCE_TYPES available
REJECTION_REASONS available


In [82]:
# Extend the validated v2 prompt with passage-level ESG requirements
def build_direct_esg_qa_prompt(row):
    return build_qa_prompt(row) + f"""

DIRECT ESG REQUIREMENT
- The question must test the ESG subject explicitly supported by the approved evidence.
- Do not create a generic administrative question merely because the parent document is ESG-related.
- The ESG connection must be understandable from the question and answer themselves.
- Do not insert ESG terminology that is absent from the evidence.
- Prefer practical compliance questions understandable to users with limited regulatory knowledge.
- If the passage is directly ESG-related but contains no obligation, a question about its definition, scope, applicability, prohibition, condition, exception, authority or legal status is permitted.
- If no directly ESG-relevant regulatory question is supported, return usable=false.

Confirmed ESG pillar: {row["final_esg_pillar"]}
"""

In [83]:
# Generate a separate three-record v3 pilot (completed IDs are skipped)
QA_PROMPT_VERSION = "2.3.5-v3-direct-esg"
GENERATION_PATH = BENCHMARK_FOLDER / f"development_qa_generated_{QA_PROMPT_VERSION}.jsonl"

completed_records = load_generation_checkpoint(GENERATION_PATH)
completed_ids = {record["evidence_id"] for record in completed_records}
pending_df = qa_evidence_df[~qa_evidence_df["evidence_id"].isin(completed_ids)].head(3)

for _, row in pending_df.iterrows():
    response = client.responses.parse(
        model=QA_MODEL,
        input=[
            {"role": "system", "content": "Generate strictly evidence-grounded direct-ESG regulatory QA data."},
            {"role": "user", "content": build_direct_esg_qa_prompt(row)},
        ],
        text_format=GeneratedQA,
    )
    result = response.output_parsed
    validate_generated_qa(result, row)
    record = {
        "qa_id": f"{row['evidence_id']}_q01", "evidence_id": row["evidence_id"],
        "doc_id": row["doc_id"], "confirmed_esg_pillar": row["final_esg_pillar"],
        "model": QA_MODEL, "prompt_version": QA_PROMPT_VERSION,
        "response_id": response.id, "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        **result.model_dump(),
    }
    with GENERATION_PATH.open("a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")

generated_df = pd.DataFrame(load_generation_checkpoint(GENERATION_PATH))
with pd.option_context("display.max_colwidth", None):
    display(generated_df[["evidence_id", "confirmed_esg_pillar", "usable", "question", "answer", "supporting_excerpts", "rejection_reason"]].tail(3))


,evidence_id,confirmed_esg_pillar,usable,question,answer,supporting_excerpts,rejection_reason
0,36_2016_TT-BTC_m_308617_article_0010,E,True,"For a petroleum contract with an annual natural resources tax period, when must the taxpayer notify the local tax agency of the provisional tax rate for expected crude oil or natural gas extraction, and when must the rate be updated if extraction estimates change significantly?","The taxpayer must determine the provisional natural resources tax rate based on the crude oil or natural gas expected to be extracted in the following year and notify the local tax agency where the tax is registered no later than 01/12 of the tax period. If changes in expected output and expected extraction days for the last six months increase or reduce the previously notified provisional rate by 15% or more, the taxpayer must determine and notify the agency of the new rate no later than 01/05 of that year.","[{'evidence_id': '36_2016_TT-BTC_m_308617_article_0010', 'excerpt': '- Where the petroleum contract has agreements on period of natural resources tax by year, based on the output of crude oil or natural gas expected to be extracted for the following year, the taxpayer shall determine the provisional natural resources tax rate and notify the local tax agency where the tax has been registered no later than 01/12 of the tax period of tax period. In the tax period, where the expected output of crude oil or natural gas and expected number of extraction days for the last 06 months have changed resulted in the increase or reduction in the provisional natural resources tax rate from 15% or more compared with the provisional natural resources tax rate which has informed to the tax agency, the taxpayers must determine and notify the new provisional natural resources tax rate to the tax agency no later than 01/05 of that year.'}]",
1,203_QD-TTg_m_532187_article_0002,E,True,What coordination is required when licensing mineral-related activities in designated areas without mineral extraction right auctions?,"The Ministry of Natural Resources and Environment must cooperate with relevant ministries and central and local authorities when licensing those mineral-related activities, in accordance with regulations.","[{'evidence_id': '203_QD-TTg_m_532187_article_0002', 'excerpt': 'Article 2. The Ministry of Natural Resources and Environment shall cooperate with relevant ministries, central and local authorities in licensing mineral-related activities in the above-mentioned areas without mineral extraction right auctions according to regulations.'}]",
2,60_2023_ND-CP_m_577202_article_0002,E,True,When do the Decree’s environmental protection inspection regulations not apply to imported motor vehicles and their parts and equipment?,They do not apply when the imported motor vehicles or imported parts and equipment are for national defense and security purposes under plans approved by the Prime Minister.,"[{'evidence_id': '60_2023_ND-CP_m_577202_article_0002', 'excerpt': '2. The regulations in this Decree do not apply to imported motor vehicles and imported parts and equipment of motor vehicles for national defense and security purposes under plans approved by the Prime Minister.'}]",


### Generate the remaining direct-ESG QA records

In [84]:
# Generate all remaining v3 records and export the review CSV
completed_ids = {record["evidence_id"] for record in load_generation_checkpoint(GENERATION_PATH)}
pending_df = qa_evidence_df[~qa_evidence_df["evidence_id"].isin(completed_ids)]

for _, row in pending_df.iterrows():
    response = client.responses.parse(
        model=QA_MODEL,
        input=[
            {"role": "system", "content": "Generate strictly evidence-grounded direct-ESG regulatory QA data."},
            {"role": "user", "content": build_direct_esg_qa_prompt(row)},
        ],
        text_format=GeneratedQA,
    )
    result = response.output_parsed
    validate_generated_qa(result, row)
    record = {
        "qa_id": f"{row['evidence_id']}_q01", "evidence_id": row["evidence_id"],
        "doc_id": row["doc_id"], "confirmed_esg_pillar": row["final_esg_pillar"],
        "model": QA_MODEL, "prompt_version": QA_PROMPT_VERSION,
        "response_id": response.id, "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        **result.model_dump(),
    }
    with GENERATION_PATH.open("a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")

generated_df = pd.DataFrame(load_generation_checkpoint(GENERATION_PATH)).drop_duplicates("evidence_id")
generated_df.to_csv(GENERATION_PATH.with_suffix(".csv"), index=False, encoding="utf-8-sig")
print(f"Complete: {len(generated_df)}/{len(qa_evidence_df)} records")
print("Review CSV:", GENERATION_PATH.with_suffix(".csv"))


Complete: 60/60 records
Review CSV: /Users/tanggiee/Desktop/RAG_AI/esg_rag_project/outputs/qa_benchmark/development_qa_generated_2.3.5-v3-direct-esg.csv


### Freeze the legal–ESG classification method

### Legal-document basis

The classification method follows Vietnam’s Law No. 64/2025/QH15 on the
Promulgation of Legislative Documents.

Under Article 7, a legislative document may contain:

`Part → Chapter → Section → Subsection → Article → Clause → Point`

Parts, Chapters, Sections, Subsections and Articles have titles. These titles
primarily provide structural context and do not automatically constitute a
complete legal rule.

Under Article 3, a legal norm is a generally applicable and binding rule of
conduct issued by a competent authority and enforced by the State.

Under Article 8, provisions that amend, supplement, replace, annul or suspend
other provisions must identify the affected document and structural unit.
Amendment provisions therefore require special evidence-boundary treatment.

### Frozen classification hierarchy

Every extracted record is classified in the following order.

#### Level 1 — Source-document authority

Record the type and promulgating authority of the source:

- Constitution;
- Law, Code or National Assembly Resolution;
- Ordinance or Standing Committee Resolution;
- Presidential Order or Decision;
- Government Decree or Resolution;
- Prime Ministerial Decision;
- Judicial Resolution;
- Circular or Joint Circular;
- provincial or district Resolution or Decision.

Document authority is metadata. It does not determine ESG relevance.

#### Level 2 — Structural evidence unit

Identify the extracted unit as:

- document;
- Part;
- Chapter;
- Section;
- Subsection;
- Article;
- Clause;
- Point; or
- embedded amended provision.

Use the lowest complete normative unit—not automatically the smallest unit.

A Clause or Point is preferred when it contains a complete rule. However, its
Article title, introductory sentence or controlling Clause must be retained as
approved context when necessary to interpret it correctly.

#### Level 3 — Evidence-boundary validity

Classify the extraction as:

- `complete`: contains a complete interpretable provision;
- `needs_parent_context`: complete only when its approved parent text is added;
- `mixed_provisions`: combines several unrelated legal provisions;
- `heading_only`: contains only a structural title;
- `incomplete`: stops mid-sentence, mid-list or before a controlling condition;
- `parser_error`: the extracted boundary does not follow the source structure.

Only `complete` and reviewed `needs_parent_context` records may proceed.

For amendment documents:

- retain the amendment lead-in and the amended provision together when both are
  necessary;
- separate different amended Articles into different evidence records;
- do not use one record containing multiple unrelated amended Articles;
- record whether the provision is amended, supplemented, replaced, annulled or
  suspended.

#### Level 4 — Legal function

Determine what the provision legally communicates:

- definition;
- scope;
- applicability;
- policy principle or target;
- right or entitlement;
- obligation;
- prohibition;
- condition;
- exception;
- procedure;
- deadline or frequency;
- authority or responsibility;
- sanction or legal consequence;
- reporting or disclosure;
- amendment or legal status.

A passage may contain more than one function, but the QA pair should normally
test one principal function.

#### Level 5 — Direct ESG relevance

A passage is `direct` only when its substantive legal function concerns an ESG
effect, risk, right, duty, target, control, disclosure, procedure, condition,
exception or consequence.

A passage is `not_direct` when ESG relevance is based only on:

- the document title;
- an Article or Chapter heading;
- the name of an environmental ministry or organization;
- an isolated ESG keyword;
- generic public administration;
- organization membership;
- an unrelated parent document; or
- incidental vocabulary.

Keywords and category labels nominate passages for review; they do not establish
direct ESG relevance.

#### Level 6 — ESG pillar

- **Environmental (E):** climate and GHG emissions; energy transition and
  efficiency; pollution; environmental quality; water; waste and circularity;
  biodiversity, ecosystems, forests and land; natural resources; environmental
  assessment, permits, remediation and compliance.

- **Social (S):** workers and employment; occupational health and safety; human
  rights; equality and non-discrimination; social insurance and welfare;
  communities and Indigenous rights; public, patient or consumer health and
  safety; access and affordability; financial inclusion; displacement and
  resettlement.

- **Governance (G):** corporate or digital governance concerning boards,
  shareholders, audit, internal controls, risk oversight, ethics,
  anti-corruption, accountability, protected reporting, corporate disclosure,
  data protection, privacy, cybersecurity, tax transparency, lobbying
  disclosure or responsible-investment stewardship.

- **MULTI:** two or more pillars independently contribute necessary substantive
  information to the same provision.

Ordinary ministry responsibilities and public administration are not Governance
by themselves.

#### Level 7 — Benchmark usability

A record is usable only when:

1. its structural boundary is valid;
2. its legal function can be identified;
3. its passage-level ESG relevance is direct;
4. its pillar can be justified from substantive content;
5. one reliable question can be answered from the approved evidence;
6. the answer does not require unavailable provisions or external assumptions;
7. it is not duplicated by a more precise evidence unit.

In [85]:
BENCHMARK_METHOD_VERSION = "2.3.5-final"
FROZEN_QA_PROMPT_VERSION = "2.3.5-v3-direct-esg"
FROZEN_QA_MODEL = "gpt-5.6"


# Vietnamese legal-document structural hierarchy under Law No. 64/2025/QH15
LEGAL_STRUCTURE_LEVELS = [
    "document",
    "part",
    "chapter",
    "section",
    "subsection",
    "article",
    "clause",
    "point",
    "embedded_amended_provision",
]


EVIDENCE_BOUNDARY_STATUSES = [
    "complete",
    "needs_parent_context",
    "mixed_provisions",
    "heading_only",
    "incomplete",
    "parser_error",
]


LEGAL_FUNCTIONS = [
    "definition",
    "scope",
    "applicability",
    "policy_principle_or_target",
    "right_or_entitlement",
    "obligation",
    "prohibition",
    "condition",
    "exception",
    "procedure",
    "deadline_or_frequency",
    "authority_or_responsibility",
    "sanction_or_legal_consequence",
    "reporting_or_disclosure",
    "amendment_or_legal_status",
]


def terms(text):
    """Convert a pipe-separated string into a clean keyword list."""
    return [term.strip() for term in text.split("|") if term.strip()]


ESG_DEFINITIONS = {
    "E": "Substantive environmental effects, risks, duties, targets, controls or disclosures concerning climate, emissions, energy transition, pollution, water, waste, biodiversity, ecosystems, forests, land, natural resources, assessment, permits, remediation or compliance.",
    "S": "Substantive effects, rights, duties or protections concerning workers, employment, occupational safety, human rights, equality, social protection, communities, public or consumer health, access, financial inclusion, displacement or resettlement.",
    "G": "Substantive corporate or digital-governance rules concerning boards, shareholders, audit, controls, risk oversight, ethics, corruption, accountability, disclosure, privacy, data protection, cybersecurity, tax transparency, lobbying disclosure or stewardship.",
}


ESG_LEXICON = {
    "E": terms("""
        climate change | climate adaptation | climate mitigation | climate resilience |
        global warming | greenhouse gas | ghg emission | carbon emission | carbon footprint |
        carbon neutrality | net zero | decarbonization | carbon credit | emissions trading |
        ozone layer | air pollution | air quality | water pollution | water quality |
        soil pollution | environmental pollution | environmental protection |
        environmental impact | environmental risk | environmental assessment |
        environmental permit | environmental license | environmental remediation |
        environmental restoration | environmental compliance | hazardous substance |
        hazardous waste | solid waste | municipal waste | medical waste | wastewater |
        sewage | waste treatment | waste management | waste collection | waste disposal |
        recycling | reuse | circular economy | extended producer responsibility |
        resource efficiency | energy efficiency | renewable energy | clean energy |
        solar power | solar energy | wind power | wind energy | hydropower | bioenergy |
        biomass | fossil fuel | coal-fired | energy transition | biodiversity | ecosystem |
        habitat | conservation | protected area | deforestation | reforestation |
        afforestation | forest protection | land degradation | natural resource |
        water resource | water withdrawal | water management | watershed | aquifer |
        drought | flood | sustainable agriculture | sustainable forestry |
        sustainable fishery | aquaculture | genetic resource | mineral extraction |
        rare earth | oil spill | chemical spill
    """),

    "S": terms("""
        human right | labour right | labor right | worker right | employee right |
        employment | working condition | working hour | minimum wage | living wage |
        equal pay | collective bargaining | trade union | freedom of association |
        forced labour | forced labor | child labour | child labor | modern slavery |
        occupational health | occupational safety | workforce health | workplace safety |
        industrial safety | personal protective equipment | hazardous work |
        employee benefit | social insurance | social security | pension | parental leave |
        maternity | discrimination | non-discrimination | gender equality |
        equal opportunity | diversity and inclusion | disability | disabled person |
        vulnerable group | indigenous people | indigenous community |
        community engagement | community health | public health | consumer health |
        consumer safety | product safety | food safety | patient safety | patient rights |
        clinical trial | access and affordability | financial inclusion |
        affordable housing | social housing | healthcare access | resettlement |
        displacement | evacuation | poverty | grievance mechanism
    """),

    "G": terms("""
        corporate governance | board of directors | board independence |
        independent director | audit committee | supervisory board | shareholder right |
        minority shareholder | shareholder meeting | proxy voting | active ownership |
        fiduciary duty | executive compensation | internal control | internal audit |
        external audit | risk management | risk oversight | compliance program |
        code of conduct | business ethics | anti-bribery | bribery | anti-corruption |
        corruption | conflict of interest | whistleblower | whistleblowing |
        protected disclosure | accountability | corporate transparency |
        beneficial owner | related-party transaction | non-financial report |
        sustainability report | esg disclosure | climate disclosure |
        environmental disclosure | integrated reporting | corporate reporting |
        supply chain due diligence | human rights due diligence | responsible investment |
        stewardship | personal data protection | data protection | data privacy | privacy |
        cybersecurity | information security | information safety | cyber risk |
        tax transparency | lobbying disclosure | political contribution disclosure
    """),
}


# Topics whose final pillar depends on the regulated impact
def conditional(pillars, rule):
    return {"possible_pillars": pillars, "rule": rule}


CONDITIONAL_ESG_TOPICS = {
    "supply chain": conditional(
        ["E", "S", "G"],
        "E for environmental impacts; S for labour or human rights; G for due diligence or oversight."
    ),
    "site closure": conditional(
        ["E", "S"],
        "E for remediation; S for worker or community impacts; MULTI when both are substantive."
    ),
    "electromagnetic field": conditional(
        ["E", "S"],
        "E for environmental impacts; S for human health or safety."
    ),
    "genetically modified organism": conditional(
        ["E", "S"],
        "E for biodiversity or ecological impacts; S for human health or food safety."
    ),
    "gmo": conditional(
        ["E", "S"],
        "E for biodiversity or ecological impacts; S for human health or food safety."
    ),
    "customer relationship": conditional(
        ["S", "G"],
        "S for consumer protection, fairness or access; G for privacy, controls or accountability."
    ),
    "philanthropy": conditional(
        ["S"],
        "S only when a substantive community right, protection or social outcome is regulated."
    ),
    "animal welfare": conditional(
        ["E"],
        "E only when biodiversity, ecosystem or conservation impacts are substantive; otherwise not_direct."
    ),
    "tax": conditional(
        ["G"],
        "G only when transparency, ethics, avoidance controls or accountability are substantive."
    ),
    "lobbying": conditional(
        ["G"],
        "G only when disclosure, political influence controls or accountability are substantive."
    ),
    "systemic risk": conditional(
        ["G"],
        "G only when risk oversight or financial-system stability is substantively regulated."
    ),
    "global compact membership": conditional(
        [],
        "Membership alone is not directly ESG-relevant."
    ),
}


print("Method version:", BENCHMARK_METHOD_VERSION)
print("Environmental terms:", len(ESG_LEXICON["E"]))
print("Social terms:", len(ESG_LEXICON["S"]))
print("Governance terms:", len(ESG_LEXICON["G"]))
print("Conditional topics:", len(CONDITIONAL_ESG_TOPICS))

Method version: 2.3.5-final
Environmental terms: 86
Social terms: 62
Governance terms: 55
Conditional topics: 12


In [86]:
# Compile literal terms into one case-insensitive regex.
# Longer terms are matched first to reduce partial matches.
def compile_terms(terms):
    cleaned = sorted({str(t).strip() for t in terms if str(t).strip()}, key=len, reverse=True)
    if not cleaned:
        return re.compile(r"(?!x)x")  # Pattern that never matches

    alternatives = [re.escape(t).replace(r"\ ", r"\s+") for t in cleaned]
    return re.compile(r"(?<!\w)(?:" + "|".join(alternatives) + r")(?!\w)", re.I)


# Compile a readable list of regex alternatives.
def compile_rules(*rules):
    return re.compile("|".join(f"(?:{rule})" for rule in rules), re.I)


# Frozen candidate patterns for each ESG pillar.
# Matches nominate passages for review; they do not determine the final label.
ESG_PATTERNS_FROZEN = {
    pillar: compile_terms(keywords)
    for pillar, keywords in ESG_LEXICON.items()
}

# Topics whose pillar depends on the regulated impact.
CONDITIONAL_TOPIC_PATTERN = compile_terms(CONDITIONAL_ESG_TOPICS)


# Remove selected agency names before keyword matching.
# Their names alone must not establish environmental relevance.
AGENCY_ONLY_PATTERN = compile_rules(
    r"\bMinistry\s+of\s+Agriculture\s+and\s+Environment\b",
    r"\bMinistry\s+of\s+Natural\s+Resources\s+and\s+Environment\b",
    r"\bDepartment\s+of\s+Natural\s+Resources\s+and\s+Environment\b",
    r"\bMinister\s+of\s+Natural\s+Resources\s+and\s+Environment\b",
)


# Signals used to identify the legal function of a passage.
# More than one function may be detected; QA generation should select one principal function.
LEGAL_FUNCTION_PATTERNS = {
    "definition": compile_rules(
        r"\bmeans\b", r"\brefers\s+to\b", r"\bshall\s+be\s+construed\s+as\b"
    ),
    "scope": compile_rules(
        r"\bthis\s+(?:constitution|law|code|ordinance|decree|resolution|decision|"
        r"circular|order)\s+(?:provides|prescribes|regulates|applies)\b"
    ),
    "applicability": compile_rules(
        r"\bappl(?:y|ies)\s+to\b", r"\bapplicable\s+to\b", r"\bregulated\s+entities\b"
    ),
    "policy_principle_or_target": compile_rules(
        r"\bprinciples?\b", r"\btargets?\b", r"\bobjectives?\b", r"\bby\s+20\d{2}\b"
    ),
    "right_or_entitlement": compile_rules(
        r"\bis\s+entitled\s+to\b", r"\bhas\s+the\s+right\s+to\b",
        r"\bshall\s+have\s+the\s+right\b"
    ),
    "obligation": compile_rules(
        r"\bshall\b(?!\s+not\b)", r"\bmust\b(?!\s+not\b)",
        r"\bis\s+required\s+to\b", r"\bis\s+responsible\s+for\b"
    ),
    "prohibition": compile_rules(
        r"\bshall\s+not\b", r"\bmust\s+not\b", r"\bmay\s+not\b",
        r"\bis\s+prohibited\b", r"\bprohibited\s+acts?\b"
    ),
    "condition": compile_rules(
        r"\bprovided\s+that\b", r"\bsubject\s+to\b", r"\bon\s+condition\s+that\b",
        r"\bunder\s+the\s+following\s+conditions\b"
    ),
    "exception": compile_rules(
        r"\bexcept(?:ion)?\b", r"\bdoes\s+not\s+apply\b",
        r"\bexempt(?:ed|ion)?\b", r"\bunless\b"
    ),
    "procedure": compile_rules(
        r"\bprocedures?\b", r"\bapplications?\b", r"\bdossiers?\b",
        r"\bsubmit(?:ted|s|ting)?\b", r"\bnotif(?:y|ies|ied|ication)\b",
        r"\bregister(?:ed|s|ing|ation)?\b"
    ),
    "deadline_or_frequency": compile_rules(
        r"\bwithin\s+\d+\s+(?:days?|working\s+days?|months?|years?)\b",
        r"\bno\s+later\s+than\b", r"\bannually\b", r"\bmonthly\b",
        r"\bquarterly\b", r"\bevery\s+\d+\b", r"\bbefore\s+\w+\s+\d{1,2}\b"
    ),
    "authority_or_responsibility": compile_rules(
        r"\bhas\s+the\s+power\s+to\b", r"\bcompetent\s+authorit(?:y|ies)\b",
        r"\bshall\s+(?:decide|organize|direct|approve|issue)\b"
    ),
    "sanction_or_legal_consequence": compile_rules(
        r"\bfines?\b", r"\bpenalt(?:y|ies)\b", r"\bsanctions?\b",
        r"\brevok(?:e|ed|ation)\b", r"\bliable\b"
    ),
    "reporting_or_disclosure": compile_rules(
        r"\breport(?:s|ed|ing)?\b", r"\bdisclos(?:e|ed|ure)\b",
        r"\bpublish(?:ed|es|ing)?\b", r"\bpublici[sz]e(?:d|s)?\b",
        r"\bprovide\s+information\b"
    ),
    "amendment_or_legal_status": compile_rules(
        r"\bamend(?:ed|ment)?\b", r"\bsupplement(?:ed|ary|ation)?\b",
        r"\breplac(?:e|ed|ement)\b", r"\bannul(?:led|ment)?\b",
        r"\bsuspend(?:ed|sion)?\b", r"\brepeal(?:ed)?\b",
        r"\btakes?\s+effect\b", r"\bceases?\s+to\s+be\s+effective\b"
    ),
}


# Broad indicators that a passage may contain a substantive legal rule.
# This is a screening signal, not final proof of benchmark usability.
NORMATIVE_SIGNAL_PATTERN = compile_rules(
    r"\bshall\b", r"\bmust\b", r"\bmeans\b", r"\bis\s+entitled\s+to\b",
    r"\bhas\s+the\s+right\b", r"\bis\s+prohibited\b", r"\bappl(?:y|ies)\s+to\b",
    r"\bshall\s+be\s+imposed\b", r"\bdoes\s+not\s+apply\b",
    r"\bis\s+required\s+to\b", r"\bis\s+responsible\s+for\b"
)


# Detect whether a passage begins with a Vietnamese legal structural label.
# A match does not mean heading_only; the remaining text must still be examined.
STRUCTURAL_HEADING_PATTERN = re.compile(
    r'^\s*["“]?(?:Part|Chapter|Section|Subsection|Article|Clause|Point)'
    r"\s+[A-Za-z0-9IVXLC.-]+\b",
    re.I,
)


print(
    f"Compiled {len(ESG_PATTERNS_FROZEN)} ESG pillar patterns, "
    f"{len(CONDITIONAL_ESG_TOPICS)} conditional topics and "
    f"{len(LEGAL_FUNCTION_PATTERNS)} legal-function patterns."
)

Compiled 3 ESG pillar patterns, 12 conditional topics and 15 legal-function patterns.


In [95]:
# Convert missing scalar values into clean strings.
def clean_text_value(value):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    return str(value).strip()


# Standardize parser labels using Vietnam's legal-document hierarchy.
def standardize_structural_level(row):
    evidence_type = clean_text_value(row.get("evidence_type")).lower()

    level_map = [
        ("embedded_article", "embedded_amended_provision"),
        ("document_fallback", "document"),
        ("subsection", "subsection"),
        ("roman_section", "section"),
        ("lettered_item", "point"),
        ("numbered_item", "clause"),
        ("section", "section"),
        ("article", "article"),
        ("clause", "clause"),
        ("point", "point"),
    ]

    return next(
        (level for marker, level in level_map if marker in evidence_type),
        "document",
    )

In [89]:
# Detect a short structural heading with no normative content.
def detect_heading_only(text):
    text = clean_text_value(text)

    return bool(
        text
        and len(text) <= 250
        and STRUCTURAL_HEADING_PATTERN.match(text)
        and not NORMATIVE_SIGNAL_PATTERN.search(text)
    )


# Screen one record for structural validity and possible legal functions.
def assess_legal_structure(row):
    text = clean_text_value(row.get("source_text"))
    structural_level = standardize_structural_level(row)

    legal_functions = [
        function
        for function, pattern in LEGAL_FUNCTION_PATTERNS.items()
        if pattern.search(text)
    ]

    # Count distinct amended Articles in one extracted record.
    amended_article_count = len(re.findall(
        r"\bArticle\s+\d+[A-Za-z.-]*\b"
        r"(?:(?!\bArticle\s+\d+).){0,150}"
        r"\b(?:amend(?:ed|ment)|supplement(?:ed|ation)|"
        r"replace(?:d|ment)|annul(?:led|ment)|suspend(?:ed|sion))\b",
        text,
        re.I | re.S,
    ))

    mixed_amendment = amended_article_count >= 2
    possible_incomplete = bool(text) and text.rstrip().endswith((",", ":", ";"))

    # Apply mutually exclusive boundary rules in priority order.
    if not text:
        boundary_status = "parser_error"
    elif detect_heading_only(text):
        boundary_status = "heading_only"
    elif mixed_amendment:
        boundary_status = "mixed_provisions"
    elif possible_incomplete:
        boundary_status = "incomplete"
    else:
        boundary_status = "complete"

    return pd.Series({
        "structural_level": structural_level,
        "boundary_status_candidate": boundary_status,
        "legal_function_candidates": json.dumps(legal_functions, ensure_ascii=False),
        "amended_article_count": amended_article_count,
        "mixed_amendment_container": mixed_amendment,
        "possible_incomplete": possible_incomplete,
    })


print("Structural assessment functions loaded.")

Structural assessment functions loaded.


In [96]:
# Screen one passage for stable ESG terms and context-dependent topics.
# Matches nominate passages for review; they do not establish direct relevance.
def assess_esg_candidate(text):
    text = clean_text_value(text)

    # An agency name alone must not create an Environmental match.
    screened_text = AGENCY_ONLY_PATTERN.sub("", text)

    stable_hits = {
        pillar: sorted({
            match.group(0).lower()
            for match in pattern.finditer(screened_text)
        })
        for pillar, pattern in ESG_PATTERNS_FROZEN.items()
    }

    conditional_hits = sorted({
        match.group(0).lower()
        for match in CONDITIONAL_TOPIC_PATTERN.finditer(screened_text)
    })

    conditional_rules = [
        {
            "topic": topic,
            "possible_pillars": CONDITIONAL_ESG_TOPICS[topic]["possible_pillars"],
            "rule": CONDITIONAL_ESG_TOPICS[topic]["rule"],
        }
        for topic in conditional_hits
        if topic in CONDITIONAL_ESG_TOPICS
    ]

    # Only stable terms nominate pillars automatically.
    # Conditional topics require contextual interpretation.
    candidate_pillars = ",".join(
        pillar
        for pillar in ("E", "S", "G")
        if stable_hits[pillar]
    )

    membership_only = (
        conditional_hits == ["global compact membership"]
        and not any(stable_hits.values())
    )

    return pd.Series({
        "candidate_esg_pillars": candidate_pillars,
        "esg_keyword_hits": json.dumps(stable_hits, ensure_ascii=False),
        "conditional_topic_hits": json.dumps(conditional_hits, ensure_ascii=False),
        "conditional_classification_rules": json.dumps(
            conditional_rules,
            ensure_ascii=False,
        ),
        "requires_contextual_classification": bool(conditional_hits),
        "membership_only": membership_only,
        "esg_hit_count": (
            sum(len(hits) for hits in stable_hits.values())
            + len(conditional_hits)
        ),
    })


print("ESG candidate assessment function loaded.")

ESG candidate assessment function loaded.


In [97]:
# Build evaluation review pool.
# The project uses development, eval and test split names.
EVALUATION_SPLIT = "eval"

evaluation_pool = evidence_candidates_df.loc[
    evidence_candidates_df["split"].eq(EVALUATION_SPLIT)
    & evidence_candidates_df["eligible"].fillna(False)
].copy()


# Stop before apply/concat if the selected split is empty.
if evaluation_pool.empty:
    raise ValueError(
        f"No eligible records found for split '{EVALUATION_SPLIT}'. "
        f"Available splits: "
        f"{evidence_candidates_df['split'].dropna().unique().tolist()}"
    )


# Remove duplicated input columns before assessment.
duplicated_columns = evaluation_pool.columns[
    evaluation_pool.columns.duplicated()
].tolist()

if duplicated_columns:
    print("Removed duplicated input columns:", duplicated_columns)
    evaluation_pool = evaluation_pool.loc[
        :, ~evaluation_pool.columns.duplicated()
    ].copy()


# Assess legal structure and possible ESG content.
structure_features = evaluation_pool.apply(
    assess_legal_structure,
    axis=1,
)

esg_features = evaluation_pool[
    "source_text"
].apply(
    assess_esg_candidate
)


# Add the generated screening features.
evaluation_pool = pd.concat(
    [
        evaluation_pool,
        structure_features,
        esg_features,
    ],
    axis=1,
)


# Confirm that feature generation did not create duplicate columns.
assert not evaluation_pool.columns.duplicated().any(), (
    "Duplicate columns remain after feature generation: "
    f"{evaluation_pool.columns[evaluation_pool.columns.duplicated()].tolist()}"
)


# Calculate passage length and assign length bands.
evaluation_pool["text_length"] = (
    evaluation_pool["source_text"]
    .fillna("")
    .astype(str)
    .str.len()
)

evaluation_pool["length_band"] = pd.cut(
    evaluation_pool["text_length"],
    bins=[0, 200, 800, 2500, float("inf")],
    labels=["short", "medium", "long", "very_long"],
    include_lowest=True,
)


# Retain structurally valid passages nominated by stable or conditional terms.
evaluation_esg_candidates = evaluation_pool.loc[
    evaluation_pool["esg_hit_count"].gt(0)
    & evaluation_pool["text_length"].ge(80)
    & evaluation_pool["boundary_status_candidate"].eq("complete")
].copy()


# Confirm that test or development records have not entered this pool.
assert evaluation_esg_candidates[
    "split"
].eq(EVALUATION_SPLIT).all()


print("Eligible evaluation evidence:", len(evaluation_pool))
print(
    "Structurally valid ESG candidates:",
    len(evaluation_esg_candidates),
)


display(
    evaluation_esg_candidates[
        "candidate_esg_pillars"
    ]
    .replace("", "CONTEXT_REVIEW")
    .value_counts()
    .rename("candidate_count")
    .to_frame()
)

display(
    evaluation_pool[
        "boundary_status_candidate"
    ]
    .value_counts(dropna=False)
    .rename("record_count")
    .to_frame()
)

Eligible evaluation evidence: 11158
Structurally valid ESG candidates: 1512


,candidate_count
candidate_esg_pillars,
E,852
S,289
G,252
CONTEXT_REVIEW,84
"E,S",22
"S,G",8
"E,G",5


,record_count
boundary_status_candidate,
complete,7840
incomplete,3239
heading_only,54
mixed_provisions,25


In [98]:
# Manual-review and final benchmark targets
EVALUATION_REVIEW_SIZE = 160
EVALUATION_TARGET_SIZE = 120
EVALUATION_RANDOM_SEED = 44


# Confirm that enough candidates are available before sampling
if len(evaluation_esg_candidates) < EVALUATION_TARGET_SIZE:
    raise ValueError(
        f"Only {len(evaluation_esg_candidates)} candidates are available, "
        f"below the final target of {EVALUATION_TARGET_SIZE}."
    )


# Use an explicit label for conditional-only records during stratification
evaluation_esg_candidates = evaluation_esg_candidates.copy()

evaluation_esg_candidates[
    "sampling_esg_pillar"
] = evaluation_esg_candidates[
    "candidate_esg_pillars"
].replace("", "CONTEXT_REVIEW")


# Sample across ESG candidates, legal structures, document types and lengths
sampling_strata = [
    "sampling_esg_pillar",
    "evidence_type",
    "document_type",
    "length_band",
]


# Select one representative from every available stratum
representatives = (
    evaluation_esg_candidates
    .groupby(
        sampling_strata,
        observed=True,
        dropna=False,
        group_keys=False,
    )
    .sample(
        n=1,
        random_state=EVALUATION_RANDOM_SEED,
    )
)


# Limit the review set or fill it with additional unused candidates
if len(representatives) >= EVALUATION_REVIEW_SIZE:
    evaluation_review_df = representatives.sample(
        n=EVALUATION_REVIEW_SIZE,
        random_state=EVALUATION_RANDOM_SEED,
    )

else:
    remaining_candidates = evaluation_esg_candidates[
        ~evaluation_esg_candidates["evidence_id"].isin(
            representatives["evidence_id"]
        )
    ]

    additional_records_needed = min(
        EVALUATION_REVIEW_SIZE - len(representatives),
        len(remaining_candidates),
    )

    additional_records = remaining_candidates.sample(
        n=additional_records_needed,
        random_state=EVALUATION_RANDOM_SEED,
    )

    evaluation_review_df = pd.concat(
        [representatives, additional_records],
        ignore_index=True,
    )


# Ensure that every evidence record appears only once
evaluation_review_df = (
    evaluation_review_df
    .drop_duplicates("evidence_id")
    .reset_index(drop=True)
)


if len(evaluation_review_df) < EVALUATION_TARGET_SIZE:
    raise ValueError(
        f"The review set contains only {len(evaluation_review_df)} records, "
        f"below the final target of {EVALUATION_TARGET_SIZE}."
    )


# Columns completed during manual legal-structure review
evaluation_review_df["final_boundary_status"] = "pending"
evaluation_review_df["primary_legal_function"] = ""


# Columns completed during manual ESG review
evaluation_review_df["esg_relevance"] = "pending"
evaluation_review_df["final_esg_pillar"] = ""
evaluation_review_df["review_notes"] = ""


EVALUATION_ESG_REVIEW_PATH = (
    BENCHMARK_FOLDER
    / "evaluation_direct_esg_review.csv"
)


# Prevent accidental loss of completed manual review
if EVALUATION_ESG_REVIEW_PATH.exists():
    raise FileExistsError(
        f"Review file already exists: {EVALUATION_ESG_REVIEW_PATH}\n"
        "Rename, move or intentionally delete it before creating a new sample."
    )


review_columns = [
    "evidence_id",
    "doc_id",
    "split",
    "document_type",
    "evidence_type",
    "evidence_number",
    "evidence_title",

    "structural_level",
    "boundary_status_candidate",
    "legal_function_candidates",
    "amended_article_count",
    "mixed_amendment_container",
    "possible_incomplete",

    "candidate_esg_pillars",
    "esg_keyword_hits",
    "conditional_topic_hits",
    "conditional_classification_rules",
    "requires_contextual_classification",

    "text_length",
    "length_band",
    "source_text",

    "final_boundary_status",
    "primary_legal_function",
    "esg_relevance",
    "final_esg_pillar",
    "review_notes",
]


# Export one fixed review sample for manual classification
evaluation_review_df[
    review_columns
].to_csv(
    EVALUATION_ESG_REVIEW_PATH,
    index=False,
    encoding="utf-8-sig",
)


print("Available ESG candidates:", len(evaluation_esg_candidates))
print("Selected for manual review:", len(evaluation_review_df))
print("Final approved target:", EVALUATION_TARGET_SIZE)
print("Review CSV:", EVALUATION_ESG_REVIEW_PATH)
print("Do not rerun this cell after editing the exported CSV.")


# Inspect whether the sampled review set has reasonable ESG coverage
display(
    evaluation_review_df[
        "sampling_esg_pillar"
    ]
    .value_counts()
    .rename("review_count")
    .to_frame()
)

display(
    evaluation_review_df[
        "length_band"
    ]
    .value_counts(sort=False)
    .rename("review_count")
    .to_frame()
)

Available ESG candidates: 1512
Selected for manual review: 160
Final approved target: 120
Review CSV: /Users/tanggiee/Desktop/RAG_AI/esg_rag_project/outputs/qa_benchmark/evaluation_direct_esg_review.csv
Do not rerun this cell after editing the exported CSV.


,review_count
sampling_esg_pillar,
E,57
S,32
CONTEXT_REVIEW,26
G,20
"E,S",17
"E,G",5
"S,G",3


,review_count
length_band,
short,24
medium,48
long,48
very_long,40


In [99]:
# Load and validate the manually reviewed evaluation evidence
EVALUATION_REVIEW_PATH = BENCHMARK_FOLDER / "evaluation_direct_esg_review.csv"

In [100]:
# Detect whether Numbers added a title row above the real column headers
first_line = EVALUATION_REVIEW_PATH.read_text(
    encoding="utf-8-sig"
).splitlines()[0]

skip_title_row = 1 if first_line.startswith("evaluation_direct_esg_review;") else 0

evaluation_review_df = pd.read_csv(
    EVALUATION_REVIEW_PATH,
    sep=";",
    skiprows=skip_title_row,
    encoding="utf-8-sig"
)

# Normalize manually entered labels
review_columns = [
    "final_boundary_status",
    "primary_legal_function",
    "esg_relevance",
    "final_esg_pillar",
    "review_notes",
]

for column in review_columns:
    evaluation_review_df[column] = (
        evaluation_review_df[column]
        .fillna("")
        .astype(str)
        .str.strip()
    )

evaluation_review_df["final_boundary_status"] = (
    evaluation_review_df["final_boundary_status"].str.lower()
)
evaluation_review_df["primary_legal_function"] = (
    evaluation_review_df["primary_legal_function"].str.lower()
)
evaluation_review_df["esg_relevance"] = (
    evaluation_review_df["esg_relevance"].str.lower()
)
evaluation_review_df["final_esg_pillar"] = (
    evaluation_review_df["final_esg_pillar"].str.upper()
)

# Frozen labels used for evaluation and test evidence
VALID_BOUNDARIES = {
    "complete",
    "needs_parent_context",
    "incomplete",
    "mixed_provisions",
}

VALID_LEGAL_FUNCTIONS = {
    "definition",
    "applicability",
    "obligation",
    "prohibition",
    "procedure",
    "condition",
    "exception",
    "right_or_entitlement",
    "authority_or_responsibility",
    "amendment_or_legal_status",
}

VALID_ESG_RELEVANCE = {"direct", "not_direct"}
VALID_ESG_PILLARS = {"E", "S", "G", "MULTI"}

# Complete passages and passages recoverable with parent context are review-eligible
evaluation_review_df["boundary_eligible"] = (
    evaluation_review_df["final_boundary_status"].isin(
        {"complete", "needs_parent_context"}
    )
)

# Identify missing or invalid manual decisions
review_issue = pd.Series("", index=evaluation_review_df.index, dtype="object")

review_issue.loc[
    ~evaluation_review_df["final_boundary_status"].isin(VALID_BOUNDARIES)
] = "invalid or missing boundary status"

review_issue.loc[
    evaluation_review_df["boundary_eligible"]
    & ~evaluation_review_df["primary_legal_function"].isin(
        VALID_LEGAL_FUNCTIONS
    )
] = "invalid or missing legal function"

review_issue.loc[
    evaluation_review_df["boundary_eligible"]
    & ~evaluation_review_df["esg_relevance"].isin(VALID_ESG_RELEVANCE)
] = "invalid or missing ESG relevance"

review_issue.loc[
    evaluation_review_df["boundary_eligible"]
    & evaluation_review_df["esg_relevance"].eq("direct")
    & ~evaluation_review_df["final_esg_pillar"].isin(VALID_ESG_PILLARS)
] = "invalid or missing ESG pillar"

evaluation_review_df["review_issue"] = review_issue
review_issues_df = evaluation_review_df[
    evaluation_review_df["review_issue"].ne("")
].copy()

# Confirm record identity before selecting evaluation evidence
assert evaluation_review_df["evidence_id"].notna().all()
assert evaluation_review_df["evidence_id"].is_unique

print("Reviewed records:", len(evaluation_review_df))
print("Direct ESG records:", evaluation_review_df["esg_relevance"].eq("direct").sum())
print(
    "Complete direct records:",
    (
        evaluation_review_df["esg_relevance"].eq("direct")
        & evaluation_review_df["final_boundary_status"].eq("complete")
    ).sum(),
)
print(
    "Direct records requiring parent context:",
    (
        evaluation_review_df["esg_relevance"].eq("direct")
        & evaluation_review_df["final_boundary_status"].eq(
            "needs_parent_context"
        )
    ).sum(),
)
print("Review issues:", len(review_issues_df))

if not review_issues_df.empty:
    display(
        review_issues_df[
            [
                "evidence_id",
                "final_boundary_status",
                "primary_legal_function",
                "esg_relevance",
                "final_esg_pillar",
                "review_issue",
            ]
        ]
    )

assert review_issues_df.empty, (
    "Resolve the displayed manual-review issue(s), save the CSV, "
    "and rerun this cell."
)

print("Manual evaluation review passed validation.")

Reviewed records: 160
Direct ESG records: 129
Complete direct records: 94
Direct records requiring parent context: 35
Review issues: 0
Manual evaluation review passed validation.


In [101]:
# Resolve parent context for direct evaluation evidence

EVALUATION_TARGET_SIZE = 120

# Retain only directly ESG-relevant passages with usable boundaries
evaluation_direct_df = evaluation_review_df[
    evaluation_review_df["esg_relevance"].eq("direct")
    & evaluation_review_df["final_boundary_status"].isin(
        {"complete", "needs_parent_context"}
    )
].copy()

# Recover parser metadata needed to locate each passage's parent structure
context_metadata_columns = [
    "evidence_id",
    "parent_unit_id",
    "evidence_start_char",
    "evidence_end_char",
]

evaluation_direct_df = evaluation_direct_df.merge(
    evidence_candidates_df[context_metadata_columns],
    on="evidence_id",
    how="left",
    validate="one_to_one",
)

# Preserve the reviewed passage as the gold evidence
evaluation_direct_df["original_evidence_text"] = (
    evaluation_direct_df["source_text"]
)

# Complete passages initially use themselves as QA evidence
evaluation_direct_df["qa_source_text"] = (
    evaluation_direct_df["source_text"]
)
evaluation_direct_df["context_source_id"] = (
    evaluation_direct_df["evidence_id"]
)
evaluation_direct_df["context_level"] = (
    evaluation_direct_df["structural_level"]
)
evaluation_direct_df["context_added"] = False
evaluation_direct_df["context_resolution_status"] = "not_required"

# Prepare Article and Clause lookups from the original parser output
article_lookup = units_df.set_index("unit_id")

clause_lookup = provisions_df[
    provisions_df["provision_type"].eq("clause")
].copy()

clause_lookup["context_length"] = (
    clause_lookup["end_char"] - clause_lookup["start_char"]
)

# Add the smallest complete parent capable of resolving each passage
context_rows = evaluation_direct_df[
    evaluation_direct_df["final_boundary_status"].eq(
        "needs_parent_context"
    )
].index

for index in context_rows:
    row = evaluation_direct_df.loc[index]

    containing_clauses = clause_lookup[
        clause_lookup["parent_unit_id"].eq(row["parent_unit_id"])
        & clause_lookup["start_char"].le(row["evidence_start_char"])
        & clause_lookup["end_char"].ge(row["evidence_end_char"])
    ]

    # A Point or item should use its containing Clause when available
    if (
        row["evidence_type"]
        in {"point", "numbered_item", "lettered_item"}
        and not containing_clauses.empty
    ):
        parent_clause = (
            containing_clauses
            .sort_values("context_length")
            .iloc[0]
        )

        evaluation_direct_df.loc[
            index,
            [
                "qa_source_text",
                "context_source_id",
                "context_level",
                "context_added",
                "context_resolution_status",
            ],
        ] = [
            parent_clause["provision_text"],
            parent_clause["provision_id"],
            "clause",
            True,
            "resolved",
        ]

    # Otherwise use the passage's complete parent Article
    elif row["parent_unit_id"] in article_lookup.index:
        parent_article = article_lookup.loc[row["parent_unit_id"]]

        evaluation_direct_df.loc[
            index,
            [
                "qa_source_text",
                "context_source_id",
                "context_level",
                "context_added",
                "context_resolution_status",
            ],
        ] = [
            parent_article["unit_text"],
            row["parent_unit_id"],
            parent_article["unit_type"],
            True,
            "resolved",
        ]

    else:
        evaluation_direct_df.loc[
            index,
            "context_resolution_status",
        ] = "unresolved"

# Verify that added context contains the complete reviewed passage
evaluation_direct_df["gold_text_preserved"] = (
    evaluation_direct_df.apply(
        lambda row: (
            row["original_evidence_text"]
            in row["qa_source_text"]
        ),
        axis=1,
    )
)

unresolved_context_df = evaluation_direct_df[
    evaluation_direct_df["context_resolution_status"].eq("unresolved")
    | ~evaluation_direct_df["gold_text_preserved"]
].copy()

In [102]:
print("Direct ESG candidates:", len(evaluation_direct_df))
print("Context not required:", (~evaluation_direct_df["context_added"]).sum())
print("Context successfully added:", evaluation_direct_df["context_added"].sum())
print("Unresolved context records:", len(unresolved_context_df))
print(
    "Available after context validation:",
    len(evaluation_direct_df) - len(unresolved_context_df),
)

if not unresolved_context_df.empty:
    display(
        unresolved_context_df[
            [
                "evidence_id",
                "evidence_type",
                "final_boundary_status",
                "parent_unit_id",
                "context_resolution_status",
                "gold_text_preserved",
            ]
        ]
    )

assert evaluation_direct_df["evidence_id"].is_unique
assert evaluation_direct_df["qa_source_text"].notna().all()
assert unresolved_context_df.empty, (
    "Review the displayed unresolved records before sampling."
)
assert len(evaluation_direct_df) >= EVALUATION_TARGET_SIZE

print("Evaluation context validation passed.")

Direct ESG candidates: 129
Context not required: 94
Context successfully added: 35
Unresolved context records: 0
Available after context validation: 129
Evaluation context validation passed.


In [103]:
# Confirm that the gold retrieval evidence was never replaced
assert (
    evaluation_direct_df["original_evidence_text"]
    == evaluation_direct_df["source_text"]
).all()

# Expanded context must contain the complete gold passage
assert evaluation_direct_df.apply(
    lambda row: (
        row["original_evidence_text"]
        in row["qa_source_text"]
    ),
    axis=1,
).all()

# Records not requiring context must remain unchanged
no_context_mask = ~evaluation_direct_df["context_added"]

assert (
    evaluation_direct_df.loc[
        no_context_mask,
        "qa_source_text",
    ]
    == evaluation_direct_df.loc[
        no_context_mask,
        "original_evidence_text",
    ]
).all()

# Context-expanded records must preserve distinct provenance
context_mask = evaluation_direct_df["context_added"]

assert (
    evaluation_direct_df.loc[
        context_mask,
        "context_source_id",
    ]
    != evaluation_direct_df.loc[
        context_mask,
        "evidence_id",
    ]
).all()

print(
    "Gold evidence preserved:",
    len(evaluation_direct_df),
    "of",
    len(evaluation_direct_df),
)
print(
    "Generation context expanded:",
    int(evaluation_direct_df["context_added"].sum()),
)

Gold evidence preserved: 129 of 129
Generation context expanded: 35


In [104]:
# Select and freeze the evaluation evidence set

EVALUATION_TARGET_SIZE = 120
EVALUATION_RANDOM_SEED = 44


In [105]:
# Preserve representation across ESG, legal and document structures
sampling_strata = [
    "final_esg_pillar",
    "primary_legal_function",
    "evidence_type",
    "document_type",
]

# Select one representative from every available stratum
stratum_representatives = (
    evaluation_direct_df
    .groupby(
        sampling_strata,
        observed=True,
        dropna=False,
        group_keys=False,
    )
    .sample(
        n=1,
        random_state=EVALUATION_RANDOM_SEED,
    )
)

assert len(stratum_representatives) <= EVALUATION_TARGET_SIZE, (
    "The number of strata exceeds the evaluation target. "
    "Reduce the sampling hierarchy before continuing."
)

In [106]:
# Fill the remaining positions from records not already selected
remaining_candidates = evaluation_direct_df[
    ~evaluation_direct_df["evidence_id"].isin(
        stratum_representatives["evidence_id"]
    )
].copy()

remaining_places = (
    EVALUATION_TARGET_SIZE - len(stratum_representatives)
)

sampled_remaining = remaining_candidates.sample(
    n=remaining_places,
    random_state=EVALUATION_RANDOM_SEED,
)

evaluation_qa_evidence_df = pd.concat(
    [stratum_representatives, sampled_remaining],
    ignore_index=True,
)

# Shuffle the final set so strata representatives are not grouped first
evaluation_qa_evidence_df = (
    evaluation_qa_evidence_df
    .sample(
        frac=1,
        random_state=EVALUATION_RANDOM_SEED,
    )
    .reset_index(drop=True)
)

# Assign stable benchmark identifiers independently of row order
evaluation_qa_evidence_df["benchmark_evidence_id"] = (
    "eval_"
    + evaluation_qa_evidence_df["evidence_id"].astype(str)
)

# Validate the frozen evaluation evidence
assert len(evaluation_qa_evidence_df) == EVALUATION_TARGET_SIZE
assert evaluation_qa_evidence_df["evidence_id"].is_unique
assert evaluation_qa_evidence_df["benchmark_evidence_id"].is_unique
assert evaluation_qa_evidence_df["split"].eq("eval").all()
assert evaluation_qa_evidence_df["esg_relevance"].eq("direct").all()
assert evaluation_qa_evidence_df["qa_source_text"].notna().all()
assert evaluation_qa_evidence_df["original_evidence_text"].notna().all()

# Confirm the gold passage remains inside the generation context
assert evaluation_qa_evidence_df.apply(
    lambda row: (
        row["original_evidence_text"]
        in row["qa_source_text"]
    ),
    axis=1,
).all()

# Save a frozen CSV for review and JSONL for reproducible generation
EVALUATION_EVIDENCE_CSV_PATH = (
    BENCHMARK_FOLDER
    / "evaluation_qa_evidence_2.3.5-final.csv"
)

EVALUATION_EVIDENCE_JSONL_PATH = (
    BENCHMARK_FOLDER
    / "evaluation_qa_evidence_2.3.5-final.jsonl"
)

evaluation_qa_evidence_df.to_csv(
    EVALUATION_EVIDENCE_CSV_PATH,
    index=False,
    encoding="utf-8-sig",
)

evaluation_qa_evidence_df.to_json(
    EVALUATION_EVIDENCE_JSONL_PATH,
    orient="records",
    lines=True,
    force_ascii=False,
)

In [107]:
# Report the frozen evidence composition
print("Eligible direct ESG records:", len(evaluation_direct_df))
print("Strata represented:", len(stratum_representatives))
print("Frozen evaluation records:", len(evaluation_qa_evidence_df))
print(
    "Records with expanded context:",
    int(evaluation_qa_evidence_df["context_added"].sum()),
)
print("CSV:", EVALUATION_EVIDENCE_CSV_PATH)
print("JSONL:", EVALUATION_EVIDENCE_JSONL_PATH)

display(
    evaluation_qa_evidence_df[
        "final_esg_pillar"
    ].value_counts().rename("record_count").to_frame()
)

display(
    pd.crosstab(
        evaluation_qa_evidence_df["primary_legal_function"],
        evaluation_qa_evidence_df["final_esg_pillar"],
        margins=True,
    )
)

display(
    pd.crosstab(
        evaluation_qa_evidence_df["evidence_type"],
        evaluation_qa_evidence_df["document_type"],
        margins=True,
    )
)

Eligible direct ESG records: 129
Strata represented: 90
Frozen evaluation records: 120
Records with expanded context: 33
CSV: /Users/tanggiee/Desktop/RAG_AI/esg_rag_project/outputs/qa_benchmark/evaluation_qa_evidence_2.3.5-final.csv
JSONL: /Users/tanggiee/Desktop/RAG_AI/esg_rag_project/outputs/qa_benchmark/evaluation_qa_evidence_2.3.5-final.jsonl


,record_count
final_esg_pillar,
E,46
MULTI,30
S,23
G,21


final_esg_pillar,E,G,MULTI,S,All
primary_legal_function,,,,,
amendment_or_legal_status,0,0,1,0,1
applicability,2,0,0,1,3
authority_or_responsibility,3,6,1,2,12
condition,2,0,0,1,3
definition,3,0,1,4,8
exception,2,0,0,1,3
obligation,21,8,16,4,49
procedure,6,6,6,3,21
prohibition,0,1,3,1,5


document_type,Circular,Decision,Decree,Integrated Document,Law,Resolution,All
evidence_type,,,,,,,
article,8,5,11,5,16,0,45
clause,11,6,11,4,10,0,42
embedded_article,0,1,1,0,3,0,5
numbered_item,0,2,0,0,0,1,3
point,6,5,8,2,2,0,23
roman_section,0,1,0,0,0,1,2
All,25,20,31,11,31,2,120


In [111]:
# Improve validation messages without changing the frozen methodology

def validate_generated_qa(result, row):
    approved_text = clean_value(row["qa_source_text"])
    expected_evidence_id = clean_value(row["evidence_id"])

    if result.usable:
        assert result.question_scope == "single_passage", (
            f"Invalid question_scope: {result.question_scope!r}"
        )

        assert result.question_type in QUESTION_TYPES, (
            f"Invalid question_type: {result.question_type!r}"
        )

        expected_group = (
            "core_compliance"
            if result.question_type in CORE_COMPLIANCE_TYPES
            else "supporting_regulatory"
        )

        assert result.compliance_group == expected_group, (
            f"Expected compliance_group={expected_group!r}, "
            f"received {result.compliance_group!r}"
        )

        assert result.question.strip(), "Question is empty."
        assert result.answer.strip(), "Answer is empty."

        assert len(result.supporting_excerpts) == 1, (
            "Exactly one supporting excerpt is required; "
            f"received {len(result.supporting_excerpts)}."
        )

        excerpt_record = result.supporting_excerpts[0]

        assert excerpt_record.evidence_id == expected_evidence_id, (
            f"Expected evidence_id={expected_evidence_id!r}, "
            f"received {excerpt_record.evidence_id!r}"
        )

        excerpt = excerpt_record.excerpt.strip()

        assert excerpt, "Supporting excerpt is empty."

        assert excerpt in approved_text, (
            "Supporting excerpt is not an exact continuous substring "
            "of qa_source_text."
        )

        assert result.rejection_reason == "", (
            "A usable result must have an empty rejection_reason."
        )

    else:
        assert result.question_scope == "", (
            "Rejected result must have an empty question_scope."
        )
        assert result.compliance_group == "", (
            "Rejected result must have an empty compliance_group."
        )
        assert result.question_type == "", (
            "Rejected result must have an empty question_type."
        )
        assert result.question == "", (
            "Rejected result must have an empty question."
        )
        assert result.answer == "", (
            "Rejected result must have an empty answer."
        )
        assert result.supporting_excerpts == [], (
            "Rejected result must not contain supporting excerpts."
        )
        assert result.rejection_reason in REJECTION_REASONS, (
            f"Invalid rejection_reason: {result.rejection_reason!r}"
        )

print("Detailed QA validator loaded.")

Detailed QA validator loaded.


In [112]:
# Generate QA pairs for the frozen evaluation evidence

# Reuse the exact model, prompt and schema frozen after development
QA_MODEL = FROZEN_QA_MODEL
QA_PROMPT_VERSION = FROZEN_QA_PROMPT_VERSION

EVALUATION_GENERATION_PATH = (
    BENCHMARK_FOLDER
    / f"evaluation_qa_generated_{QA_PROMPT_VERSION}.jsonl"
)

# Load completed records so interrupted runs resume safely
completed_records = load_generation_checkpoint(
    EVALUATION_GENERATION_PATH
)

completed_ids = {
    record["evidence_id"]
    for record in completed_records
}

assert len(completed_ids) == len(completed_records), (
    "Duplicate evidence IDs exist in the evaluation checkpoint."
)

# Submit only evidence that has not already been generated
pending_evaluation_df = evaluation_qa_evidence_df[
    ~evaluation_qa_evidence_df["evidence_id"].isin(completed_ids)
].copy()

print("Previously completed:", len(completed_ids))
print("Remaining records:", len(pending_evaluation_df))

for number, (_, row) in enumerate(
    pending_evaluation_df.iterrows(),
    start=1,
):
    try:
        response = client.responses.parse(
            model=QA_MODEL,
            input=[
                {
                    "role": "system",
                    "content": (
                        "Generate strictly evidence-grounded ESG "
                        "regulatory-compliance QA data using the "
                        "frozen schema and methodology."
                    ),
                },
                {
                    "role": "user",
                    "content": build_direct_esg_qa_prompt(row),
                },
            ],
            text_format=GeneratedQA,
        )

        result = response.output_parsed

        assert result is not None, (
            "No structured output was returned."
        )

        # Apply the same automatic checks used in development
        validate_generated_qa(result, row)

        record = {
            "qa_id": f"{row['evidence_id']}_q01",
            "benchmark_evidence_id": row["benchmark_evidence_id"],
            "evidence_id": row["evidence_id"],
            "doc_id": row["doc_id"],
            "split": "eval",
            "model": QA_MODEL,
            "prompt_version": QA_PROMPT_VERSION,
            "response_id": response.id,
            "generated_at_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            **result.model_dump(),
        }

        # Save each successful result immediately
        with EVALUATION_GENERATION_PATH.open(
            "a",
            encoding="utf-8",
        ) as file:
            file.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )

        completed_ids.add(row["evidence_id"])

        print(
            f"[{number}/{len(pending_evaluation_df)}] saved | "
            f"{row['evidence_id']} | "
            f"{result.question_type} | "
            f"usable={result.usable}"
        )

        time.sleep(0.3)

    except Exception as error:
        error_text = str(error)

        print(
            f"[{number}/{len(pending_evaluation_df)}] ERROR | "
            f"{row['evidence_id']} | "
            f"{type(error).__name__}: {error}"
        )

        # Stop when credits are unavailable; the checkpoint is preserved
        if (
            "credit_balance_exhausted" in error_text
            or "insufficient_quota" in error_text
        ):
            print(
                "Generation stopped because API credits "
                "are unavailable."
            )
            break

        # Pause briefly after temporary request-limit errors
        if (
            "429" in error_text
            or "rate limit" in error_text.lower()
        ):
            time.sleep(15)

# Reload and validate the complete checkpoint
evaluation_generated_records = load_generation_checkpoint(
    EVALUATION_GENERATION_PATH
)

evaluation_generated_df = pd.DataFrame(
    evaluation_generated_records
).drop_duplicates(
    "evidence_id",
    keep="first",
)

remaining_ids = (
    set(evaluation_qa_evidence_df["evidence_id"])
    - set(evaluation_generated_df["evidence_id"])
)

assert evaluation_generated_df["evidence_id"].is_unique
assert set(evaluation_generated_df["evidence_id"]) <= set(
    evaluation_qa_evidence_df["evidence_id"]
)

# Export a readable copy without changing the checkpoint
EVALUATION_GENERATION_CSV_PATH = (
    EVALUATION_GENERATION_PATH.with_suffix(".csv")
)

evaluation_generated_df.to_csv(
    EVALUATION_GENERATION_CSV_PATH,
    index=False,
    encoding="utf-8-sig",
)

Previously completed: 119
Remaining records: 1
[1/1] ERROR | 43_2025_TT-BCT_m_674734_article_0005 | AssertionError: Supporting excerpt is not an exact continuous substring of qa_source_text.


In [109]:
print()
print(
    "Completed:",
    len(evaluation_generated_df),
    "/",
    len(evaluation_qa_evidence_df),
)
print("Still unfinished:", len(remaining_ids))

if not evaluation_generated_df.empty:
    print(
        "Usable:",
        int(evaluation_generated_df["usable"].sum()),
    )
    print(
        "Rejected by generator:",
        int((~evaluation_generated_df["usable"]).sum()),
    )

print("Checkpoint:", EVALUATION_GENERATION_PATH)
print("CSV:", EVALUATION_GENERATION_CSV_PATH)

if remaining_ids:
    display(
        pd.DataFrame(
            {"unfinished_evidence_id": sorted(remaining_ids)}
        )
    )
else:
    print("Evaluation QA generation is complete.")


Completed: 119 / 120
Still unfinished: 1
Usable: 119
Rejected by generator: 0
Checkpoint: /Users/tanggiee/Desktop/RAG_AI/esg_rag_project/outputs/qa_benchmark/evaluation_qa_generated_2.3.5-v3-direct-esg.jsonl
CSV: /Users/tanggiee/Desktop/RAG_AI/esg_rag_project/outputs/qa_benchmark/evaluation_qa_generated_2.3.5-v3-direct-esg.csv


,unfinished_evidence_id
0,43_2025_TT-BCT_m_674734_article_0005


In [115]:
# Replace the excluded record with the closest unused record

EXCLUDED_EVIDENCE_ID = (
    "43_2025_TT-BCT_m_674734_article_0005"
)

# Recover the record being excluded
excluded_rows_df = evaluation_qa_evidence_df[
    evaluation_qa_evidence_df["evidence_id"].eq(
        EXCLUDED_EVIDENCE_ID
    )
].copy()

assert len(excluded_rows_df) == 1, (
    "The excluded evidence ID was not found exactly once."
)

excluded_record = excluded_rows_df.iloc[0]

# Identify the nine eligible records not included in the frozen sample
unused_records_df = evaluation_direct_df[
    ~evaluation_direct_df["evidence_id"].isin(
        evaluation_qa_evidence_df["evidence_id"]
    )
].copy()

assert len(unused_records_df) == 9, (
    f"Expected 9 unused records, found {len(unused_records_df)}."
)

# Score similarity to preserve the most important distributions
# ESG pillar and legal function receive greater weight
unused_records_df["replacement_match_score"] = (
    unused_records_df["final_esg_pillar"]
    .eq(excluded_record["final_esg_pillar"])
    .astype(int) * 4
    + unused_records_df["primary_legal_function"]
    .eq(excluded_record["primary_legal_function"])
    .astype(int) * 4
    + unused_records_df["evidence_type"]
    .eq(excluded_record["evidence_type"])
    .astype(int) * 2
    + unused_records_df["document_type"]
    .eq(excluded_record["document_type"])
    .astype(int)
)

# Select deterministically from the highest-scoring candidates
replacement_record_df = (
    unused_records_df
    .sort_values(
        ["replacement_match_score", "evidence_id"],
        ascending=[False, True],
    )
    .head(1)
    .drop(columns="replacement_match_score")
    .copy()
)

replacement_evidence_id = replacement_record_df.iloc[0][
    "evidence_id"
]

# Replace only the problematic record
evaluation_qa_evidence_df = pd.concat(
    [
        evaluation_qa_evidence_df[
            ~evaluation_qa_evidence_df["evidence_id"].eq(
                EXCLUDED_EVIDENCE_ID
            )
        ],
        replacement_record_df,
    ],
    ignore_index=True,
)

# Reconfirm the stable benchmark identifier
evaluation_qa_evidence_df["benchmark_evidence_id"] = (
    "eval_"
    + evaluation_qa_evidence_df["evidence_id"].astype(str)
)

# Audit the replacement
assert len(evaluation_qa_evidence_df) == 120
assert evaluation_qa_evidence_df["evidence_id"].is_unique
assert EXCLUDED_EVIDENCE_ID not in set(
    evaluation_qa_evidence_df["evidence_id"]
)
assert replacement_evidence_id in set(
    evaluation_qa_evidence_df["evidence_id"]
)

print("Excluded:", EXCLUDED_EVIDENCE_ID)
print("Replacement:", replacement_evidence_id)

display(
    pd.DataFrame([
        {
            "record": "excluded",
            **excluded_record[sampling_strata].to_dict(),
        },
        {
            "record": "replacement",
            **replacement_record_df.iloc[0][
                sampling_strata
            ].to_dict(),
        },
    ])
)

Excluded: 43_2025_TT-BCT_m_674734_article_0005
Replacement: 152_2020_ND-CP_m_461585_article_0027


,record,final_esg_pillar,primary_legal_function,evidence_type,document_type
0,excluded,S,obligation,article,Circular
1,replacement,S,authority_or_responsibility,article,Decree


In [116]:
# Generate QA pairs for the frozen evaluation evidence

# Reuse the exact model, prompt and schema frozen after development
QA_MODEL = FROZEN_QA_MODEL
QA_PROMPT_VERSION = FROZEN_QA_PROMPT_VERSION

EVALUATION_GENERATION_PATH = (
    BENCHMARK_FOLDER
    / f"evaluation_qa_generated_{QA_PROMPT_VERSION}.jsonl"
)

# Load completed records so interrupted runs resume safely
completed_records = load_generation_checkpoint(
    EVALUATION_GENERATION_PATH
)

completed_ids = {
    record["evidence_id"]
    for record in completed_records
}

assert len(completed_ids) == len(completed_records), (
    "Duplicate evidence IDs exist in the evaluation checkpoint."
)

# Submit only evidence that has not already been generated
pending_evaluation_df = evaluation_qa_evidence_df[
    ~evaluation_qa_evidence_df["evidence_id"].isin(completed_ids)
].copy()

print("Previously completed:", len(completed_ids))
print("Remaining records:", len(pending_evaluation_df))

for number, (_, row) in enumerate(
    pending_evaluation_df.iterrows(),
    start=1,
):
    try:
        response = client.responses.parse(
            model=QA_MODEL,
            input=[
                {
                    "role": "system",
                    "content": (
                        "Generate strictly evidence-grounded ESG "
                        "regulatory-compliance QA data using the "
                        "frozen schema and methodology."
                    ),
                },
                {
                    "role": "user",
                    "content": build_direct_esg_qa_prompt(row),
                },
            ],
            text_format=GeneratedQA,
        )

        result = response.output_parsed

        assert result is not None, (
            "No structured output was returned."
        )

        # Apply the same automatic checks used in development
        validate_generated_qa(result, row)

        record = {
            "qa_id": f"{row['evidence_id']}_q01",
            "benchmark_evidence_id": row["benchmark_evidence_id"],
            "evidence_id": row["evidence_id"],
            "doc_id": row["doc_id"],
            "split": "eval",
            "model": QA_MODEL,
            "prompt_version": QA_PROMPT_VERSION,
            "response_id": response.id,
            "generated_at_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            **result.model_dump(),
        }

        # Save each successful result immediately
        with EVALUATION_GENERATION_PATH.open(
            "a",
            encoding="utf-8",
        ) as file:
            file.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )

        completed_ids.add(row["evidence_id"])

        print(
            f"[{number}/{len(pending_evaluation_df)}] saved | "
            f"{row['evidence_id']} | "
            f"{result.question_type} | "
            f"usable={result.usable}"
        )

        time.sleep(0.3)

    except Exception as error:
        error_text = str(error)

        print(
            f"[{number}/{len(pending_evaluation_df)}] ERROR | "
            f"{row['evidence_id']} | "
            f"{type(error).__name__}: {error}"
        )

        # Stop when credits are unavailable; the checkpoint is preserved
        if (
            "credit_balance_exhausted" in error_text
            or "insufficient_quota" in error_text
        ):
            print(
                "Generation stopped because API credits "
                "are unavailable."
            )
            break

        # Pause briefly after temporary request-limit errors
        if (
            "429" in error_text
            or "rate limit" in error_text.lower()
        ):
            time.sleep(15)

# Reload and validate the complete checkpoint
evaluation_generated_records = load_generation_checkpoint(
    EVALUATION_GENERATION_PATH
)

evaluation_generated_df = pd.DataFrame(
    evaluation_generated_records
).drop_duplicates(
    "evidence_id",
    keep="first",
)

remaining_ids = (
    set(evaluation_qa_evidence_df["evidence_id"])
    - set(evaluation_generated_df["evidence_id"])
)

assert evaluation_generated_df["evidence_id"].is_unique
assert set(evaluation_generated_df["evidence_id"]) <= set(
    evaluation_qa_evidence_df["evidence_id"]
)

# Export a readable copy without changing the checkpoint
EVALUATION_GENERATION_CSV_PATH = (
    EVALUATION_GENERATION_PATH.with_suffix(".csv")
)

evaluation_generated_df.to_csv(
    EVALUATION_GENERATION_CSV_PATH,
    index=False,
    encoding="utf-8-sig",
)

Previously completed: 119
Remaining records: 1
[1/1] saved | 152_2020_ND-CP_m_461585_article_0027 | obligation | usable=True


In [117]:
print()
print(
    "Completed:",
    len(evaluation_generated_df),
    "/",
    len(evaluation_qa_evidence_df),
)
print("Still unfinished:", len(remaining_ids))

if not evaluation_generated_df.empty:
    print(
        "Usable:",
        int(evaluation_generated_df["usable"].sum()),
    )
    print(
        "Rejected by generator:",
        int((~evaluation_generated_df["usable"]).sum()),
    )

print("Checkpoint:", EVALUATION_GENERATION_PATH)
print("CSV:", EVALUATION_GENERATION_CSV_PATH)

if remaining_ids:
    display(
        pd.DataFrame(
            {"unfinished_evidence_id": sorted(remaining_ids)}
        )
    )
else:
    print("Evaluation QA generation is complete.")


Completed: 120 / 120
Still unfinished: 0
Usable: 120
Rejected by generator: 0
Checkpoint: /Users/tanggiee/Desktop/RAG_AI/esg_rag_project/outputs/qa_benchmark/evaluation_qa_generated_2.3.5-v3-direct-esg.jsonl
CSV: /Users/tanggiee/Desktop/RAG_AI/esg_rag_project/outputs/qa_benchmark/evaluation_qa_generated_2.3.5-v3-direct-esg.csv
Evaluation QA generation is complete.


In [120]:
# Freeze the final reviewed evaluation QA benchmark
FINAL_EVALUATION_SIZE = 118

REVIEWED_EVALUATION_QA_PATH = (
    BENCHMARK_FOLDER
    / "evaluation_qa_generated_2.3.5-v3-direct-esg.csv"
)

In [123]:
# Detect whether Numbers added a title row above the real headers
first_line = REVIEWED_EVALUATION_QA_PATH.read_text(
    encoding="utf-8-sig"
).splitlines()[0]

skip_title_row = (
    1
    if first_line.startswith(
        "evaluation_qa_generated_2.3.5-v3-direct-esg;"
    )
    else 0
)


# Load the final manually reviewed QA file
evaluation_qa_reviewed_df = pd.read_csv(
    REVIEWED_EVALUATION_QA_PATH,
    sep=";",
    skiprows=skip_title_row,
    encoding="utf-8-sig",
)


# Normalize text fields without changing their substantive content
reviewed_text_columns = [
    "qa_id",
    "benchmark_evidence_id",
    "evidence_id",
    "doc_id",
    "split",
    "model",
    "prompt_version",
    "response_id",
    "generated_at_utc",
    "question_scope",
    "compliance_group",
    "question_type",
    "question",
    "answer",
    "rejection_reason",
]

for column in reviewed_text_columns:
    evaluation_qa_reviewed_df[column] = (
        evaluation_qa_reviewed_df[column]
        .fillna("")
        .astype(str)
        .str.strip()
    )


# Normalize Boolean values exported by Numbers
evaluation_qa_reviewed_df["usable"] = (
    evaluation_qa_reviewed_df["usable"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "true": True,
        "false": False,
    })
)

assert evaluation_qa_reviewed_df["usable"].notna().all(), (
    "The usable column contains an unrecognized value."
)


# Parse and standardize the serialized supporting excerpts
evaluation_qa_reviewed_df[
    "supporting_excerpts_parsed"
] = evaluation_qa_reviewed_df[
    "supporting_excerpts"
].apply(
    ast.literal_eval
)

evaluation_qa_reviewed_df["supporting_excerpts"] = (
    evaluation_qa_reviewed_df[
        "supporting_excerpts_parsed"
    ].apply(
        lambda value: json.dumps(
            value,
            ensure_ascii=False,
        )
    )
)


# Validate the reviewed QA records
assert len(evaluation_qa_reviewed_df) == FINAL_EVALUATION_SIZE
assert evaluation_qa_reviewed_df["qa_id"].is_unique
assert evaluation_qa_reviewed_df["evidence_id"].is_unique
assert evaluation_qa_reviewed_df["question"].is_unique

assert evaluation_qa_reviewed_df["usable"].all()
assert evaluation_qa_reviewed_df[
    "question_scope"
].eq("single_passage").all()

assert evaluation_qa_reviewed_df["question"].ne("").all()
assert evaluation_qa_reviewed_df["answer"].ne("").all()


# Confirm that reviewed records form a valid subset of frozen evidence
reviewed_evidence_ids = set(
    evaluation_qa_reviewed_df["evidence_id"]
)

frozen_evidence_ids = set(
    evaluation_qa_evidence_df["evidence_id"]
)

assert reviewed_evidence_ids <= frozen_evidence_ids, (
    "The reviewed QA file contains an evidence ID that is not "
    "present in the frozen evidence set."
)

assert len(
    frozen_evidence_ids - reviewed_evidence_ids
) == 2, (
    "Exactly two duplicate-question evidence records should "
    "have been removed."
)


# Retain only evidence corresponding to the 118 reviewed questions
final_evaluation_evidence_df = (
    evaluation_qa_evidence_df[
        evaluation_qa_evidence_df["evidence_id"].isin(
            reviewed_evidence_ids
        )
    ]
    .copy()
)

assert len(final_evaluation_evidence_df) == FINAL_EVALUATION_SIZE


# Select the evidence metadata required for retrieval evaluation
gold_evidence_columns = [
    "evidence_id",
    "benchmark_evidence_id",
    "doc_id",
    "document_type",
    "evidence_type",
    "evidence_number",
    "evidence_title",
    "structural_level",
    "final_boundary_status",
    "primary_legal_function",
    "final_esg_pillar",
    "original_evidence_text",
    "qa_source_text",
    "context_source_id",
    "context_level",
    "context_added",
]


# Merge each reviewed QA pair with its immutable gold evidence
evaluation_benchmark_df = (
    evaluation_qa_reviewed_df
    .drop(
        columns=[
            "benchmark_evidence_id",
            "doc_id",
        ]
    )
    .merge(
        final_evaluation_evidence_df[
            gold_evidence_columns
        ],
        on="evidence_id",
        how="left",
        validate="one_to_one",
    )
)


# Extract the exact supporting excerpt into a separate field
evaluation_benchmark_df["supporting_excerpt"] = (
    evaluation_benchmark_df[
        "supporting_excerpts_parsed"
    ].apply(
        lambda excerpts: excerpts[0]["excerpt"].strip()
    )
)


# Validate supporting-evidence identity
evaluation_benchmark_df[
    "supporting_excerpt_id_valid"
] = evaluation_benchmark_df.apply(
    lambda row: (
        row["supporting_excerpts_parsed"][0][
            "evidence_id"
        ]
        == row["evidence_id"]
    ),
    axis=1,
)


# Confirm the excerpt occurs in the text used for QA generation
evaluation_benchmark_df[
    "excerpt_in_qa_source"
] = evaluation_benchmark_df.apply(
    lambda row: (
        row["supporting_excerpt"]
        in row["qa_source_text"]
    ),
    axis=1,
)


# Record whether the excerpt is entirely inside the gold passage
evaluation_benchmark_df[
    "excerpt_in_gold_evidence"
] = evaluation_benchmark_df.apply(
    lambda row: (
        row["supporting_excerpt"]
        in row["original_evidence_text"]
    ),
    axis=1,
)


# Apply final integrity checks
assert len(evaluation_benchmark_df) == FINAL_EVALUATION_SIZE
assert evaluation_benchmark_df["qa_id"].is_unique
assert evaluation_benchmark_df["evidence_id"].is_unique
assert evaluation_benchmark_df["question"].is_unique

assert evaluation_benchmark_df[
    "original_evidence_text"
].notna().all()

assert evaluation_benchmark_df[
    "qa_source_text"
].notna().all()

assert evaluation_benchmark_df[
    "supporting_excerpt_id_valid"
].all()

assert evaluation_benchmark_df[
    "excerpt_in_qa_source"
].all(), (
    "At least one supporting excerpt is not an exact substring "
    "of qa_source_text."
)


# Remove the temporary parsed Python objects before export
export_evaluation_benchmark_df = (
    evaluation_benchmark_df
    .drop(columns="supporting_excerpts_parsed")
    .copy()
)


# Save the immutable final evaluation benchmark
FINAL_EVALUATION_CSV_PATH = (
    BENCHMARK_FOLDER
    / "evaluation_qa_benchmark_2.3.5-final.csv"
)

FINAL_EVALUATION_JSONL_PATH = (
    BENCHMARK_FOLDER
    / "evaluation_qa_benchmark_2.3.5-final.jsonl"
)

export_evaluation_benchmark_df.to_csv(
    FINAL_EVALUATION_CSV_PATH,
    index=False,
    encoding="utf-8-sig",
)

export_evaluation_benchmark_df.to_json(
    FINAL_EVALUATION_JSONL_PATH,
    orient="records",
    lines=True,
    force_ascii=False,
)

In [124]:
# Report the final benchmark audit
print("Final evaluation QA pairs:", len(evaluation_benchmark_df))
print(
    "Unique questions:",
    evaluation_benchmark_df["question"].nunique(),
)
print(
    "Supporting excerpts in QA source:",
    int(
        evaluation_benchmark_df[
            "excerpt_in_qa_source"
        ].sum()
    ),
)
print(
    "Supporting excerpts entirely in gold evidence:",
    int(
        evaluation_benchmark_df[
            "excerpt_in_gold_evidence"
        ].sum()
    ),
)
print(
    "Questions using expanded context:",
    int(
        evaluation_benchmark_df["context_added"].sum()
    ),
)
print("Final CSV:", FINAL_EVALUATION_CSV_PATH)
print("Final JSONL:", FINAL_EVALUATION_JSONL_PATH)
print("Final evaluation QA benchmark frozen.")

Final evaluation QA pairs: 118
Unique questions: 118
Supporting excerpts in QA source: 118
Supporting excerpts entirely in gold evidence: 116
Questions using expanded context: 33
Final CSV: /Users/tanggiee/Desktop/RAG_AI/esg_rag_project/outputs/qa_benchmark/evaluation_qa_benchmark_2.3.5-final.csv
Final JSONL: /Users/tanggiee/Desktop/RAG_AI/esg_rag_project/outputs/qa_benchmark/evaluation_qa_benchmark_2.3.5-final.jsonl
Final evaluation QA benchmark frozen.


In [125]:
# Inspect QA pairs whose excerpts extend beyond gold evidence

context_dependent_qa_df = evaluation_benchmark_df[
    ~evaluation_benchmark_df[
        "excerpt_in_gold_evidence"
    ]
].copy()


print(
    "QA pairs requiring gold-grounding review:",
    len(context_dependent_qa_df),
)


display(
    context_dependent_qa_df[
        [
            "qa_id",
            "evidence_id",
            "evidence_type",
            "question",
            "answer",
            "supporting_excerpt",
            "original_evidence_text",
            "qa_source_text",
            "context_source_id",
            "context_level",
        ]
    ]
)

QA pairs requiring gold-grounding review: 2


,qa_id,evidence_id,evidence_type,question,answer,supporting_excerpt,original_evidence_text,qa_source_text,context_source_id,context_level
12,219_2025_ND-CP_m_669168_article_0007_clause_00...,219_2025_ND-CP_m_669168_article_0007_clause_003,clause,Under what condition is a foreign chairperson ...,The person is exempt if their capital contribu...,Article 7. Foreign workers exempt from the req...,3. Chairpersons of the Board of Directors or m...,Article 7. Foreign workers exempt from the req...,219_2025_ND-CP_m_669168_article_0007,article
73,25_2018_TT-BNNPTNT_m_411295_article_0008_claus...,25_2018_TT-BNNPTNT_m_411295_article_0008_claus...,clause,What must a risk assessment of imported live a...,The risk assessment must cover: compliance wit...,Article 8. Contents of risk assessment of impo...,1. Conformity with regulations concerning prod...,Article 8. Contents of risk assessment of impo...,25_2018_TT-BNNPTNT_m_411295_article_0008,article


In [126]:
# Display each context-dependent QA pair in a full vertical view

from IPython.display import display, Markdown
import pandas as pd


# Prevent pandas from truncating long legal text
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)


review_fields = [
    "qa_id",
    "evidence_id",
    "evidence_type",
    "question",
    "answer",
    "supporting_excerpt",
    "original_evidence_text",
    "qa_source_text",
    "context_source_id",
    "context_level",
]


for review_number, (_, row) in enumerate(
    context_dependent_qa_df.iterrows(),
    start=1,
):
    display(
        Markdown(
            f"## Gold-grounding review {review_number} "
            f"of {len(context_dependent_qa_df)}"
        )
    )

    # Show fields vertically so long text is not compressed
    record_view_df = pd.DataFrame({
        "field": review_fields,
        "value": [
            row[field]
            for field in review_fields
        ],
    })

    display(
        record_view_df.style.set_properties(
            subset=["field"],
            **{
                "font-weight": "bold",
                "width": "220px",
                "vertical-align": "top",
            },
        ).set_properties(
            subset=["value"],
            **{
                "white-space": "pre-wrap",
                "text-align": "left",
                "vertical-align": "top",
                "min-width": "900px",
            },
        )
    )

## Gold-grounding review 1 of 2

,field,value
0,qa_id,219_2025_ND-CP_m_669168_article_0007_clause_003_q01
1,evidence_id,219_2025_ND-CP_m_669168_article_0007_clause_003
2,evidence_type,clause
3,question,Under what condition is a foreign chairperson or member of a Board of Directors in a limited liability company exempt from the work permit requirement?
4,answer,The person is exempt if their capital contribution is valued at VND 3 billion or more.
5,supporting_excerpt,"Article 7. Foreign workers exempt from the requirement for a work permit 1. Persons who fall under one of the cases specified in Clauses 3, 4, 5, 6, 7, and 8 of Article 154 of the Labor Code. 2. Owners or capital contributors with a capital contribution value of VND 3 billion or more in a limited liability company. 3. Chairpersons of the Board of Directors or members of the Board of Directors with a capital contribution value of VND 3 billion or more in a limited liability company."
6,original_evidence_text,3. Chairpersons of the Board of Directors or members of the Board of Directors with a capital contribution value of VND 3 billion or more in a limited liability company.
7,qa_source_text,"Article 7. Foreign workers exempt from the requirement for a work permit 1. Persons who fall under one of the cases specified in Clauses 3, 4, 5, 6, 7, and 8 of Article 154 of the Labor Code. 2. Owners or capital contributors with a capital contribution value of VND 3 billion or more in a limited liability company. 3. Chairpersons of the Board of Directors or members of the Board of Directors with a capital contribution value of VND 3 billion or more in a limited liability company. 4. Persons who enter Vietnam to provide consulting services on expertise and technology or to perform other tasks serving the research, development, appraisal, monitoring, evaluation, management, and implementation of programs or projects using official development assistance in accordance with regulations or agreements in international treaties on official development assistance signed between competent authorities of Vietnam and foreign countries. 5. Foreign journalists engaging in press activities confirmed by the Ministry of Foreign Affairs. 6. Persons who are sent by competent foreign agency or organization to Vietnam to teach or work as a manager, or CEO at educational institutions established upon the request of foreign diplomatic missions, intergovernmental organizations in Vietnam, or institutions/organizations established under international treaties to which Vietnam is a signatory. 7. Foreign students or trainees studying at educational institutions in Vietnam or foreign countries who have an internship agreement or a job offer letter from an employer in Vietnam; or interns or apprentices on a Vietnam sea-going ship. 8. Family of members of foreign representative bodies in Vietnam who are authorized to work in Vietnam under international treaties to which the Socialist Republic of Vietnam is a signatory. 9. Persons who hold an official passport and work for state agencies, political organizations, or socio-political organizations. 10. Persons who are responsible for establishing a commercial presence. 11. Volunteers who do voluntary and unpaid works in Vietnam to execute international treaties to which the Socialist Republic of Vietnam is a signatory and have confirmation from a foreign diplomatic mission or international organization in Vietnam. 12. Persons who enter Vietnam to execute international agreements signed by central or provincial agencies or organizations under the law. 13. Foreign workers who are managers, CEOs, experts, or technical workers who fall under one of the following cases: a) Entering Vietnam to work for a total period of less than 90 days within one year, from January 1 to December 31; b) Under intra-company transfer program; Person who is an intra-company transferee of a foreign enterprise that has established a commercial presence in Vietnam within 11 sectors in the schedule of commitments in 

## Gold-grounding review 2 of 2

,field,value
0,qa_id,25_2018_TT-BNNPTNT_m_411295_article_0008_clause_001_q01
1,evidence_id,25_2018_TT-BNNPTNT_m_411295_article_0008_clause_001
2,evidence_type,clause
3,question,What must a risk assessment of imported live aquatic animals and plants cover?
4,answer,"The risk assessment must cover: compliance with product-safety regulations for live aquatic animals and plants imported for food; their ability to survive, grow, develop, and compete with native aquatic species for food in the Vietnamese environment; their ability to become harmful or potentially harmful species and reproduce in that environment; their ability to hybridize naturally with native aquatic species; and the risk of spreading pathogens to native aquatic species and humans."
5,supporting_excerpt,"Article 8. Contents of risk assessment of imported live aquatic animals and plants 1. Conformity with regulations concerning product safety applied to import of live aquatic animals and plants for food. 2. Ability to survive, grow and develop in the Vietnamese environment and ability to compete against native aquatic species for food. 3. Ability to become harmful species or potentially harmful species and reproduction ability in the Vietnamese environment. 4. Ability to hybridize between the imported aquatic species and native aquatic species under natural conditions. 5. Risk of spreading pathogens to native aquatic species and human."
6,original_evidence_text,1. Conformity with regulations concerning product safety applied to import of live aquatic animals and plants for food.
7,qa_source_text,"Article 8. Contents of risk assessment of imported live aquatic animals and plants 1. Conformity with regulations concerning product safety applied to import of live aquatic animals and plants for food. 2. Ability to survive, grow and develop in the Vietnamese environment and ability to compete against native aquatic species for food. 3. Ability to become harmful species or potentially harmful species and reproduction ability in the Vietnamese environment. 4. Ability to hybridize between the imported aquatic species and native aquatic species under natural conditions. 5. Risk of spreading pathogens to native aquatic species and human."
8,context_source_id,25_2018_TT-BNNPTNT_m_411295_article_0008
9,context_level,article


Record 1: keep the question and answer; shorten its excerpt to the original gold clause.

Record 2: narrow the question and answer to item 1 because only item 1 is gold evidence.

In [127]:
# Correct the two QA pairs so every answer is supported by gold evidence
def correct_qa(qa_id, question=None, answer=None):
    index = evaluation_benchmark_df.index[
        evaluation_benchmark_df["qa_id"].eq(qa_id)
    ][0]

    gold_text = evaluation_benchmark_df.at[
        index, "original_evidence_text"
    ].strip()

    if question is not None:
        evaluation_benchmark_df.at[index, "question"] = question

    if answer is not None:
        evaluation_benchmark_df.at[index, "answer"] = answer

    evaluation_benchmark_df.at[
        index, "supporting_excerpt"
    ] = gold_text

    evaluation_benchmark_df.at[
        index, "supporting_excerpts"
    ] = json.dumps(
        [{
            "evidence_id": evaluation_benchmark_df.at[
                index, "evidence_id"
            ],
            "excerpt": gold_text,
        }],
        ensure_ascii=False,
    )

In [128]:
# Record 1: only shorten the supporting excerpt
correct_qa(
    "219_2025_ND-CP_m_669168_article_0007_clause_003_q01"
)


# Record 2: narrow the QA pair to the approved gold clause
correct_qa(
    "25_2018_TT-BNNPTNT_m_411295_article_0008_clause_001_q01",
    question=(
        "What product-safety requirement must a risk assessment "
        "cover for live aquatic animals and plants imported for food?"
    ),
    answer=(
        "It must assess conformity with the product-safety "
        "regulations applicable to live aquatic animals and "
        "plants imported for food."
    ),
)

In [129]:
# Confirm that every excerpt is now inside its gold evidence
evaluation_benchmark_df["excerpt_in_gold_evidence"] = (
    evaluation_benchmark_df.apply(
        lambda row: (
            row["supporting_excerpt"]
            in row["original_evidence_text"]
        ),
        axis=1,
    )
)

assert len(evaluation_benchmark_df) == 118
assert evaluation_benchmark_df["question"].is_unique
assert evaluation_benchmark_df[
    "excerpt_in_gold_evidence"
].all()

In [130]:
# Save the corrected final benchmark
export_df = evaluation_benchmark_df.drop(
    columns=["supporting_excerpts_parsed"],
    errors="ignore",
)

export_df.to_csv(
    FINAL_EVALUATION_CSV_PATH,
    index=False,
    encoding="utf-8-sig",
)

export_df.to_json(
    FINAL_EVALUATION_JSONL_PATH,
    orient="records",
    lines=True,
    force_ascii=False,
)

print("Final QA pairs:", len(export_df))
print("Gold-grounded excerpts:", export_df[
    "excerpt_in_gold_evidence"
].sum())
print("Final benchmark corrected and saved.")

Final QA pairs: 118
Gold-grounded excerpts: 118
Final benchmark corrected and saved.
